In [1]:
from google.colab import drive
drive.mount('/content/drive')
%cd /content/drive/MyDrive/Projects/karate-kumite-performance-profiles-lab/

Mounted at /content/drive
/content/drive/MyDrive/Projects/karate-kumite-performance-profiles-lab


In [2]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from pathlib import Path

from scipy.cluster.hierarchy import linkage, fcluster
from sklearn.cluster import KMeans, DBSCAN
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import (
    silhouette_score, davies_bouldin_score,
    adjusted_rand_score, normalized_mutual_info_score,
)

RNG_SEED = 0
np.random.seed(RNG_SEED)

# Exploratory Data Analysis (EDA)

In [3]:
DATA_DIR = Path("data")
OUTPUT_DIR = Path("outputs")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

FILES = {
    "Female-42kg": DATA_DIR / "female_42kg.csv",
    "Female-47kg": DATA_DIR / "female_47kg.csv",
    "Female-54kg": DATA_DIR / "female_54kg.csv",
}

P0_FEATURES = ["yuko", "waza_ari", "ippon", "attack", "defend",
               "longest_point_in_sec", "fastest_point_in_sec"]

TIMING_FEATURES    = ["longest_point_in_sec", "fastest_point_in_sec"]
NONTIMING_FEATURES = [c for c in P0_FEATURES if c not in TIMING_FEATURES]

S1_FEATURES = P0_FEATURES + ["match_attack", "match_defend"]
S2_FEATURES = P0_FEATURES + ["state_numeric", "hantei", "point_diff"]
STRUCTURAL_ZERO_FEATURES = ["yuko", "longest_point_in_sec", "fastest_point_in_sec"]


In [4]:
def load_weight_class(name, filepath):
    if filepath.exists():
        df = pd.read_csv(filepath, sep=";")
        df = df.dropna(how="all").reset_index(drop=True)
        df.columns = df.columns.str.strip()
    else:
        print(f"{filepath} not found")
        return None

    df["state_numeric"] = df["state"].str.lower().map({"win": 1, "lose": 0})
    return df

In [5]:
# @title Load Data
raw_data = {name: load_weight_class(name, path) for name, path in FILES.items()}

for name, df in raw_data.items():
  print("Data: ", name, " with ", df.shape[0], " samples and ", df.shape[1], " features")

print("")
print("Features: ")
for cfg_name, feats in [("P0", P0_FEATURES), ("S1", S1_FEATURES), ("S2", S2_FEATURES)]:
  print(cfg_name, ": ", feats)

print("")
print("Samples [P0 features only]: ")
display(raw_data["Female-42kg"][["name"] + P0_FEATURES].head(3))

Data:  Female-42kg  with  38  samples and  21  features
Data:  Female-47kg  with  38  samples and  21  features
Data:  Female-54kg  with  38  samples and  21  features

Features: 
P0 :  ['yuko', 'waza_ari', 'ippon', 'attack', 'defend', 'longest_point_in_sec', 'fastest_point_in_sec']
S1 :  ['yuko', 'waza_ari', 'ippon', 'attack', 'defend', 'longest_point_in_sec', 'fastest_point_in_sec', 'match_attack', 'match_defend']
S2 :  ['yuko', 'waza_ari', 'ippon', 'attack', 'defend', 'longest_point_in_sec', 'fastest_point_in_sec', 'state_numeric', 'hantei', 'point_diff']

Samples [P0 features only]: 


,name,yuko,waza_ari,ippon,attack,defend,longest_point_in_sec,fastest_point_in_sec
0,S. Mosleh,2,0,2,4,2,6,2
1,Vinogradova,0,0,0,4,1,0,0
2,Abellanosa,0,0,0,2,4,0,0


# Pre-processing

## P0, Two-part Encoding (TP), No-timing score (NT) and Score Only (SO)

In [6]:
#@title Genearl Function
def build_matrix(df, feature_list):
    X_raw     = df[feature_list].apply(pd.to_numeric, errors="raise")
    variances = X_raw.var(ddof=0)
    zero_var  = variances[variances == 0].index.tolist()
    retained  = [c for c in feature_list if c not in zero_var]
    X_final   = X_raw[retained]
    scaler    = StandardScaler()
    X_scaled  = scaler.fit_transform(X_final)
    return X_scaled, retained, zero_var

In [7]:
#@title Two-part Encoding Function
def build_two_part_matrix(df):
    """
    Two-part encoding of the timing variables:
      Part 1: timing_event = 1 if a scoring-time event was recorded, else 0
      Part 2: each timing variable z-scored using SCORERS ONLY;
              non-scorers set to 0 (= the average scorer)
    Zero-variance removal is reused from build_matrix().
    """
    timing = df[TIMING_FEATURES].apply(pd.to_numeric, errors="raise")
    scored = (timing > 0).any(axis=1).to_numpy()

    # Part 1: non-timing features + event indicator (goes through build_matrix)
    df_tmp = df.copy()
    df_tmp["timing_event"] = scored.astype(float)
    X_part1, retained, dropped = build_matrix(
        df_tmp, NONTIMING_FEATURES + ["timing_event"]
    )

    # Part 2: conditional durations
    cond_cols, cond_names = [], []
    for c in TIMING_FEATURES:
        s = timing.loc[scored, c]
        if s.std(ddof=0) == 0:              # constant among scorers -> drop
            dropped.append(c + "_cond")
            continue
        z = ((timing[c] - s.mean()) / s.std(ddof=0)).to_numpy(dtype=float, copy=True)
        z[~scored] = 0.0
        cond_cols.append(z)
        cond_names.append(c + "_cond")

    X_scaled = np.column_stack([X_part1] + cond_cols)
    return X_scaled, retained + cond_names, dropped

In [8]:
#@title Display Pre-processed Data Shape

prepared = {}
for wc, df in raw_data.items():
    prepared[wc] = {}
    scored   = (df[TIMING_FEATURES] > 0).any(axis=1).to_numpy()
    all_rows = np.ones(len(df), dtype=bool)

    # Original configurations
    for cfg_name, feats in [("P0", P0_FEATURES), ("S1", S1_FEATURES), ("S2", S2_FEATURES)]:
        Xs, retained, dropped = build_matrix(df, feats)
        prepared[wc][cfg_name] = {"X": Xs, "features": retained, "dropped": dropped, "rows": all_rows}
        print(wc, cfg_name, "->", Xs.shape, "dropped:", dropped)

    # TP: two-part encoding
    Xs, retained, dropped = build_two_part_matrix(df)
    prepared[wc]["TP"] = {"X": Xs, "features": retained, "dropped": dropped, "rows": all_rows}
    print(wc, "TP", "->", Xs.shape, "dropped:", dropped)

    # NT: no timing variables
    Xs, retained, dropped = build_matrix(df, NONTIMING_FEATURES)
    prepared[wc]["NT"] = {"X": Xs, "features": retained, "dropped": dropped, "rows": all_rows}
    print(wc, "NT", "->", Xs.shape, "dropped:", dropped)

    # SO: scorers only (fewer rows)
    df_so = df.loc[scored].reset_index(drop=True)
    Xs, retained, dropped = build_matrix(df_so, P0_FEATURES)
    prepared[wc]["SO"] = {"X": Xs, "features": retained, "dropped": dropped, "rows": scored}
    print(wc, "SO", "->", Xs.shape, "dropped:", dropped)
    print("")

Female-42kg P0 -> (38, 7) dropped: []
Female-42kg S1 -> (38, 9) dropped: []
Female-42kg S2 -> (38, 10) dropped: []
Female-42kg TP -> (38, 8) dropped: []
Female-42kg NT -> (38, 5) dropped: []
Female-42kg SO -> (24, 7) dropped: []

Female-47kg P0 -> (38, 7) dropped: []
Female-47kg S1 -> (38, 9) dropped: []
Female-47kg S2 -> (38, 9) dropped: ['hantei']
Female-47kg TP -> (38, 8) dropped: []
Female-47kg NT -> (38, 5) dropped: []
Female-47kg SO -> (29, 7) dropped: []

Female-54kg P0 -> (38, 5) dropped: ['waza_ari', 'ippon']
Female-54kg S1 -> (38, 7) dropped: ['waza_ari', 'ippon']
Female-54kg S2 -> (38, 8) dropped: ['waza_ari', 'ippon']
Female-54kg TP -> (38, 6) dropped: ['waza_ari', 'ippon']
Female-54kg NT -> (38, 3) dropped: ['waza_ari', 'ippon']
Female-54kg SO -> (25, 5) dropped: ['waza_ari', 'ippon']



## Shared metrics, admissibility, and the selection protocol

In [9]:
#@title Constant
K_VALUES = [2, 3, 4, 5]
MIN_CLUSTER_SIZE = 4

KMEANS_SEEDS = list(range(100))
KMEANS_N_INIT = 20
RSKC_SEEDS = list(range(100))
ARSK_SEEDS = list(range(100))
STOF_CONTAMINATION_PERCENT = 2.0

DBSCAN_EPS_VALUES = np.round(np.arange(0.3, 4.01, 0.1), 1)
DBSCAN_MINPTS_VALUES = [4, 5, 6]
DBSCAN_MIN_RETENTION = 0.80
EPS_NUM = 1e-12

In [10]:
#@title Metrics
def cluster_sizes(labels, ignore_noise=False):
    vals, counts = np.unique(labels, return_counts=True)
    if ignore_noise:
        keep = vals != -1
        vals, counts = vals[keep], counts[keep]
    return dict(zip(vals.tolist(), counts.tolist()))


def min_cluster_size(labels, ignore_noise=False):
    sizes = cluster_sizes(labels, ignore_noise=ignore_noise)
    return min(sizes.values()) if sizes else 0


def safe_metrics(X, labels, ignore_noise=False):
    """
    The safe_metrics function won't crash on a degenerate partition

    scikit-learn's silhouette_score and davies_bouldin_score raise an error
    if the labels contain fewer than 2 distinct clusters.
    """

    mask = labels != -1 if ignore_noise else np.ones(len(labels), dtype=bool)
    Xu, lu = X[mask], labels[mask]
    if len(set(lu)) < 2:
        return np.nan, np.nan
    return silhouette_score(Xu, lu), davies_bouldin_score(Xu, lu)


def wcss_bcss_ratio(X, labels):
    mask = labels != -1
    Xu, lu = X[mask], labels[mask]
    grand_mean = Xu.mean(axis=0)
    wcss, bcss = 0.0, 0.0
    for c in np.unique(lu):
        pts = Xu[lu == c]
        cm = pts.mean(axis=0)
        wcss += ((pts - cm) ** 2).sum()
        bcss += len(pts) * ((cm - grand_mean) ** 2).sum()
    return wcss / bcss if bcss > EPS_NUM else np.nan


In [11]:
#@title Addmisible: Ward's Hierarchical Clustering

def fit_ward_all_k(X, k_values=K_VALUES, min_size=MIN_CLUSTER_SIZE):
    Z = linkage(X, method="ward", metric="euclidean")
    candidates = []
    for k in k_values:
        labels = fcluster(Z, t=k, criterion="maxclust") - 1
        sil, dbi = safe_metrics(X, labels)
        admissible = (len(set(labels)) == k and min_cluster_size(labels) >= min_size
                      and not np.isnan(sil))
        candidates.append({"k": k, "labels": labels, "silhouette": sil, "dbi": dbi,
                            "wcss_bcss": wcss_bcss_ratio(X, labels), "admissible": admissible})
    adm = [c for c in candidates if c["admissible"]]
    if not adm:
        return None, candidates
    return max(adm, key=lambda c: c["silhouette"]), candidates


In [12]:
#@title Addmisible: k-Means, RSKC k-Means, and ARSK k-Means

def select_representative_run(admissible_runs, target_median_silhouette):
    """
    Final partition = admissible run whose silhouette is CLOSEST TO THE
    (LOWER) MEDIAN admissible silhouette for that k (not the maximum).
    Ties -> lower DBI, then lower WCSS/BCSS, then lower seed.
    """
    tmp = admissible_runs.copy()
    tmp["distance_to_median"] = np.abs(tmp["silhouette"] - target_median_silhouette)
    tmp = tmp.sort_values(
        by=["distance_to_median", "dbi", "wcss_bcss", "seed"],
        ascending=[True, True, True, True],
    )
    return tmp.iloc[0]


def fit_across_seeds(X, fit_one_fn, seeds, k_values=K_VALUES, min_size=MIN_CLUSTER_SIZE):
    """
    Generic seed/parameter sweep + median-silhouette selection.
    Used IDENTICALLY for k-Means, RSKC-STOF, and ARSK so all three
    "seeded" methods are held to the same protocol.
    fit_one_fn(X, k, seed) -> integer label array
    """

    runs = []
    for k in k_values:
        for seed in seeds:
            labels = fit_one_fn(X, k, seed)
            sil, dbi = safe_metrics(X, labels)
            admissible = (
                len(set(labels)) == k
                and min_cluster_size(labels) >= min_size
                and not np.isnan(sil)
            )
            runs.append({"k": k, "seed": seed, "labels": labels, "silhouette": sil,
                         "dbi": dbi, "wcss_bcss": wcss_bcss_ratio(X, labels),
                         "admissible": admissible})
    runs_df = pd.DataFrame(runs)

    selected_per_k = {}
    for k in k_values:
        adm = runs_df[(runs_df.k == k) & (runs_df.admissible)]
        if len(adm) == 0:
            continue
        # Lower median: an OBSERVED silhouette value. With an even number of
        # runs, the ordinary median is the midpoint of two runs, both equally
        # close to it, and floating-point noise decides which is chosen.
        target = np.sort(adm["silhouette"].to_numpy())[(len(adm) - 1) // 2]
        selected_per_k[k] = select_representative_run(adm, target)
    if not selected_per_k:
        return None, runs_df

    best_k = max(selected_per_k, key=lambda kk: selected_per_k[kk]["silhouette"])
    return selected_per_k[best_k], runs_df


# Algorithms

In [13]:
#@title k-Means

def kmeans_once_wrapper(X, k, seed):
    return KMeans(n_clusters=k, n_init=KMEANS_N_INIT, random_state=seed).fit_predict(X)

In [14]:
#@title RSKC-STOF

# ============================================================
# ROBUST SPARSE CLUSTERING METHODS
#   1. RSKC-STOF  (Madjoukeng, Kenmogne & Frenay, 2025, IJDSA)
#   2. ARSK       (Li, Sugasawa & Katayama, 2024, TMLR)
#
# Both are independent re-implementations from the published
# algorithms (the authors' code repositories were not publicly
# accessible). Choices not specified in the papers are marked
# "[UNSPECIFIED IN PAPER]" and must be reported in the manuscript.
#
# Both plug into fit_across_seeds(X, fn, seeds) through a factory:
#     fit_across_seeds(X, make_rskc_wrapper(X), RSKC_SEEDS)
#     fit_across_seeds(X, make_arsk_wrapper(X), ARSK_SEEDS)
# The factory tunes the sparsity/robustness parameters ONCE per k
# (not once per seed) and caches them.
# ============================================================

import numpy as np
from sklearn.cluster import kmeans_plusplus

EPS_NUM = 1e-12


# ============================================================
# Shared helpers
# ============================================================

def bcss_per_feature(X, labels):
    """Between-cluster sum of squares for each feature (TSS - WSS)."""
    X = np.asarray(X, float)
    tss = ((X - X.mean(axis=0)) ** 2).sum(axis=0)
    wss = np.zeros(X.shape[1])
    for c in np.unique(labels):
        m = labels == c
        wss += ((X[m] - X[m].mean(axis=0)) ** 2).sum(axis=0)
    return np.maximum(tss - wss, 0.0)


def weighted_lloyd(X, k, weights, centers, max_iter=300, tol=1e-4):
    """
    k-Means with weighted squared distance sum_j w_j (x_j - z_j)^2.
    Equivalent to running k-Means on features multiplied by sqrt(w_j).
    Starts from the given centers.
    """
    X = np.asarray(X, float)
    centers = np.asarray(centers, float).copy()
    prev = None
    for _ in range(max_iter):
        d2 = (weights[None, None, :] * (X[:, None, :] - centers[None, :, :]) ** 2).sum(axis=2)
        labels = d2.argmin(axis=1)

        # Empty cluster -> re-seed with the point farthest from its center
        missing = [c for c in range(k) if c not in set(labels)]
        if missing:
            far = d2.min(axis=1).copy()
            for c in missing:
                i = int(far.argmax())
                centers[c] = X[i]
                far[i] = -np.inf
            d2 = (weights[None, None, :] * (X[:, None, :] - centers[None, :, :]) ** 2).sum(axis=2)
            labels = d2.argmin(axis=1)

        new = centers.copy()
        for c in range(k):
            if (labels == c).any():
                new[c] = X[labels == c].mean(axis=0)
        shift = np.linalg.norm(new - centers)
        centers = new
        if (prev is not None and np.array_equal(labels, prev)) or shift < tol:
            break
        prev = labels.copy()
    return labels, centers


# ============================================================
# 1. RSKC-STOF
# ============================================================
# Paper -> code mapping
#   Eq. 1 sparse k-means: ||w||_2 = 1, ||w||_1 <= s, w >= 0
#   Alg. 1 RKAS   : first center random; next centers by D^2
#                   sampling; candidate rejected if STOF w.r.t. the
#                   WHOLE dataset                -> rkas_initialization
#   Alg. 2 ESD    : per cluster, per feature, leave-one-out
#                   contribution (x_j - z_j)^2 / sum_{others}(x_ij - z_j)^2,
#                   STOF if > (100 - p)th percentile of the
#                   other observations' contributions -> detect_stof_esd
#   Alg. 3        : w_j = 1/sqrt(d); weighted k-means (sqrt(w) scaling);
#                   ESD; weights from gamma'_j computed WITHOUT the STOF
#                   values; STOFs are NOT removed; stop when relative
#                   weight change < 1e-4               -> rskc_stof_once
#   Eq. 2 weights : soft-threshold gamma' by Delta, then L2-normalise
#
# [UNSPECIFIED IN PAPER] and our choice:
#   - Delta: binary search so that ||w||_1 = s (Witten & Tibshirani, 2010)
#   - s: permutation gap statistic (Witten & Tibshirani, 2010),
#        10 candidates in [1.2, 0.9*sqrt(d)], 25 permutations
#   - ESD skipped for clusters with < 4 observations
#     (leave-one-out percentile not meaningful)
# ============================================================

def sparse_weight_update(gamma, s):
    """w = S(gamma, Delta)/||S(gamma, Delta)||_2 with ||w||_1 <= s."""
    gamma = np.maximum(np.asarray(gamma, float), 0.0)
    d = len(gamma)
    if np.all(gamma <= EPS_NUM):
        return np.ones(d) / np.sqrt(d)
    w0 = gamma / np.linalg.norm(gamma)
    if w0.sum() <= s:                       # constraint already met: Delta = 0
        return w0
    lo, hi, best = 0.0, gamma.max(), w0
    for _ in range(100):
        mid = (lo + hi) / 2
        cand = np.maximum(gamma - mid, 0.0)
        nrm = np.linalg.norm(cand)
        if nrm <= EPS_NUM:
            hi = mid
            continue
        cand /= nrm
        best = cand
        if abs(cand.sum() - s) < 1e-8:
            break
        lo, hi = (mid, hi) if cand.sum() > s else (lo, mid)
    return best


def _safe_ratio(num, den):
    if den <= EPS_NUM:
        return 0.0 if num <= EPS_NUM else np.inf
    return num / den


def _row_percentile(A, q):
    """
    np.percentile(..., method="linear") along axis 1 ignoring NaN,
    reproducing numpy's interpolation formula exactly.
    A: (m, m, d). Returns (m, d); NaN where a row has no finite values.
    """
    S = np.sort(A, axis=1)                       # NaN sorted to the end
    cnt = np.isfinite(A).sum(axis=1)             # (m, d)
    pos = (q / 100.0) * (cnt - 1)
    lo = np.floor(pos).astype(int)
    hi = np.ceil(pos).astype(int)
    lo_c = np.clip(lo, 0, A.shape[1] - 1)
    hi_c = np.clip(hi, 0, A.shape[1] - 1)
    a = np.take_along_axis(S, lo_c[:, None, :], axis=1)[:, 0, :]
    b = np.take_along_axis(S, hi_c[:, None, :], axis=1)[:, 0, :]
    t = pos - lo
    diff = b - a
    out = np.where(t >= 0.5, b - diff * (1 - t), a + diff * t)
    return np.where(cnt > 0, out, np.nan)


def detect_stof_esd(X, labels, contamination_percent=2.0):
    """
    Algorithm 2, vectorised. Gives identical decisions to the loop
    version _detect_stof_esd_loop (verified on 1,800 random partitions),
    about 25x faster:
    for each cluster (m >= 4), feature j and observation i, compare i's
    leave-one-out contribution with the (100-p)th percentile of the other
    observations' contributions.
    """
    X = np.asarray(X, float)
    n, d = X.shape
    mask = np.zeros((n, d), bool)
    q = 100.0 - contamination_percent
    for c in np.unique(labels):
        idx = np.where(labels == c)[0]
        m = len(idx)
        if m < 4:
            continue
        V = X[idx]                                   # (m, d)
        Z = (V.sum(axis=0)[None, :] - V) / (m - 1)   # leave-one-out centre, row li
        R = (V[None, :, :] - Z[:, None, :]) ** 2     # (li, r, d)
        eye = np.eye(m, dtype=bool)[:, :, None]
        R = np.where(eye, 0.0, R)
        tot = R.sum(axis=1)                          # (li, d)
        num_x = (V - Z) ** 2                         # (li, d)
        contrib = np.where(tot <= EPS_NUM,
                           np.where(num_x <= EPS_NUM, 0.0, np.inf),
                           num_x / np.where(tot <= EPS_NUM, 1.0, tot))
        den = tot[:, None, :] - R
        ref = np.where(den <= EPS_NUM,
                       np.where(R <= EPS_NUM, 0.0, np.inf),
                       R / np.where(den <= EPS_NUM, 1.0, den))
        ref = np.where(eye | ~np.isfinite(ref), np.nan, ref)
        thr = _row_percentile(ref, q)                # (li, d)
        flag = contrib > thr
        flag &= ~np.isnan(thr)
        mask[idx] = flag
    return mask


def _detect_stof_esd_loop(X, labels, contamination_percent=2.0):
    """Reference (loop) version of Algorithm 2, kept for verification."""
    X = np.asarray(X, float)
    n, d = X.shape
    mask = np.zeros((n, d), bool)
    q = 100.0 - contamination_percent
    for c in np.unique(labels):
        idx = np.where(labels == c)[0]
        m = len(idx)
        if m < 4:
            continue
        for j in range(d):
            v = X[idx, j]
            for li in range(m):
                others = v[np.arange(m) != li]
                z = others.mean()
                res = (others - z) ** 2
                tot = res.sum()
                contrib_x = _safe_ratio((v[li] - z) ** 2, tot)
                ref = np.array([_safe_ratio(r, tot - r) for r in res])
                ref = ref[np.isfinite(ref)]
                if len(ref) and contrib_x > np.percentile(ref, q):
                    mask[idx[li], j] = True
    return mask


def rkas_initialization(X, k, seed, contamination_percent=2.0):
    """Algorithm 1 (robust k-means++ against STOF observations)."""
    X = np.asarray(X, float)
    n = len(X)
    rng = np.random.default_rng(seed)
    global_stof = detect_stof_esd(X, np.zeros(n, int), contamination_percent).any(axis=1)

    selected = [int(rng.integers(0, n))]
    attempts = 0
    while len(selected) < k and attempts < 5000:
        sel = np.asarray(selected)
        d2 = ((X[:, None, :] - X[sel][None, :, :]) ** 2).sum(axis=2).min(axis=1)
        d2[sel] = 0.0
        if d2.sum() <= EPS_NUM:
            break
        cand = int(rng.choice(n, p=d2 / d2.sum()))
        attempts += 1
        if cand not in selected and not global_stof[cand]:
            selected.append(cand)

    # Fallback (only if almost every candidate is a STOF): farthest non-STOF point
    while len(selected) < k:
        rest = [i for i in range(n) if i not in selected]
        sel = np.asarray(selected)
        d2 = ((X[rest][:, None, :] - X[sel][None, :, :]) ** 2).sum(axis=2).min(axis=1)
        ok = [p for p, i in enumerate(rest) if not global_stof[i]] or list(range(len(rest)))
        selected.append(rest[max(ok, key=lambda p: d2[p])])

    return X[selected[:k]].copy()


def robust_gamma(X, labels, stof_mask):
    """gamma'_j: BCSS of feature j using only its non-STOF values."""
    X = np.asarray(X, float)
    d = X.shape[1]
    g = np.zeros(d)
    for j in range(d):
        ok = ~stof_mask[:, j]
        if ok.sum() < 2:
            continue
        xj = X[ok, j]
        tss = ((xj - xj.mean()) ** 2).sum()
        wss = 0.0
        for c in np.unique(labels):
            v = X[(labels == c) & ok, j]
            if len(v):
                wss += ((v - v.mean()) ** 2).sum()
        g[j] = max(tss - wss, 0.0)
    return g


def rskc_stof_once(X, k, s, seed, contamination_percent=2.0, max_iter=300, tol=1e-4):
    """Algorithm 3. Returns dict with labels, weights, stof_mask."""
    X = np.asarray(X, float)
    n, d = X.shape
    w = np.ones(d) / np.sqrt(d)
    centers = rkas_initialization(X, k, seed, contamination_percent)
    converged = False
    for it in range(1, max_iter + 1):
        w_old = w.copy()
        labels, centers = weighted_lloyd(X, k, w, centers)
        stof = detect_stof_esd(X, labels, contamination_percent)
        w = sparse_weight_update(robust_gamma(X, labels, stof), s)
        if np.abs(w - w_old).sum() / max(np.abs(w_old).sum(), EPS_NUM) < tol:
            converged = True
            break
    labels, centers = weighted_lloyd(X, k, w, centers)
    stof = detect_stof_esd(X, labels, contamination_percent)
    return {"labels": labels, "weights": w, "stof_mask": stof,
            "iterations": it, "converged": converged}


# ---- sparsity bound s: permutation gap -----------------------

def _sparse_kmeans_objective(X, k, s, seed, max_iter=100, tol=1e-4):
    """Non-robust sparse k-means, used only to choose s."""
    d = X.shape[1]
    w = np.ones(d) / np.sqrt(d)
    centers, _ = kmeans_plusplus(X, n_clusters=k, random_state=seed)
    for _ in range(max_iter):
        w_old = w.copy()
        labels, centers = weighted_lloyd(X, k, w, centers)
        w = sparse_weight_update(bcss_per_feature(X, labels), s)
        if np.abs(w - w_old).sum() / max(np.abs(w_old).sum(), EPS_NUM) < tol:
            break
    labels, centers = weighted_lloyd(X, k, w, centers)
    return max(float((w * bcss_per_feature(X, labels)).sum()), EPS_NUM)


def select_s_by_permutation(X, k, n_permutations=25, n_candidates=10, seed=2026):
    X = np.asarray(X, float)
    d = X.shape[1]
    rng = np.random.default_rng(seed)
    upper, lower = 0.9 * np.sqrt(d), 1.2
    if upper <= lower:
        lower, upper = 1.01, max(1.02, 0.99 * np.sqrt(d))
    s_values = np.exp(np.linspace(np.log(lower), np.log(upper), n_candidates))

    obs = np.array([_sparse_kmeans_objective(X, k, s, seed + i) for i, s in enumerate(s_values)])
    perm = np.zeros((n_candidates, n_permutations))
    for b in range(n_permutations):
        Xb = np.column_stack([rng.permutation(X[:, j]) for j in range(d)])
        for i, s in enumerate(s_values):
            perm[i, b] = _sparse_kmeans_objective(Xb, k, s, seed + 1000 + b * 100 + i)
    gaps = np.log(obs) - np.log(perm).mean(axis=1)
    return float(s_values[int(gaps.argmax())]), s_values, gaps


def make_rskc_wrapper(X, contamination_percent=2.0, tuning_seed=2026, verbose=True):
    """
    Returns fn(X, k, seed) for fit_across_seeds.
    s is selected once per k (permutation gap) and cached in fn.s_by_k.
    """
    X = np.asarray(X, float)
    cache = {}

    def fn(X_in, k, seed):
        if k not in cache:
            s, _, _ = select_s_by_permutation(X, k, seed=tuning_seed + 10 * k)
            cache[k] = s
            if verbose:
                print(f"    RSKC-STOF: k={k}, selected s={s:.4f}")
        return rskc_stof_once(X_in, k, cache[k], seed, contamination_percent)["labels"]

    fn.s_by_k = cache
    return fn

In [15]:
#@title ARSK
# ============================================================
# 2. ARSK
# ============================================================
# Paper -> code mapping (arXiv:2407.06945, Algorithms 1 and 2)
#   Objective  : sum_j w_j Q_j^R(X - E) - sum_i P1(||E_i||_2; lam1)
#                - sum_j P2(w_j; lam2),   ||w||_2 = 1
#   Q_j^R      : between-cluster scatter of feature j on X - E
#                (pairwise form = 2 x BCSS)
#   E update   : group lasso (Eq. 7), row-wise:
#                E_i = r_i * max(0, 1 - lam1/||r_i||), r_i = x*_i - mu_k
#                computed on the weighted data, then restored
#                E := E / sqrt(w_j)                    (Alg. 1, step 3c)
#   w update   : soft-threshold Q^R by lam2, then L2-normalise (Eq. 9)
#   Stopping   : sum|w_new - w| / sum|w| < eps
#   Tuning     : robust Gap statistic (Eq. 10), B = 25 permutations,
#                Algorithm 2: fix lam1 = lam1_dagger, choose lam2;
#                then fix lam2*, choose lam1.
#   Version    : soft (group lasso) for E and soft for w. The paper
#                reserves SCAD for ultra-high-dimensional data; with
#                5-7 features soft/soft is the natural choice.
#
# [UNSPECIFIED IN PAPER] and our choice:
#   - Weighted data: sqrt(w_j) scaling (consistent with step 3c and
#     with sparse k-means). The paper writes w_j(X - E) in step 3a but
#     divides by sqrt(w_j) in step 3c.
#   - Initial E: paper sets E = X for "the 80% of data farthest from all
#     data center points". With n = 38 this collapses 30 observations
#     onto one point at the start and changes the final partition, so we
#     start from E = 0 (INIT_FRAC = 0.0); 0.8 is kept as an option.
#   - Grids ("exponential decay", values not given): 10 values each.
#       lam2: from 0.9 x max_j Q_j down to 1% of it
#       lam1: from max row residual norm down to its 20th percentile
#     (Q_j and residuals from an ordinary k-means fit at the same k)
#   - lam1_dagger = max row residual norm (E = 0, i.e. no trimming),
#     so lam2 is chosen first for plain sparse clustering, then lam1.
#   - Permutation: each feature permuted independently (Witten &
#     Tibshirani, 2010); the paper says "randomly selecting from the
#     original dataset".
# ============================================================

# Initial E (Alg. 1, step 2). The paper's literal rule ("E = X for the 80%
# of data farthest from all data center points") collapses 30 of 38
# observations onto the same point in the first clustering step and makes
# the result depend on that artefact (checked on this data). We therefore
# start from E = 0 (no initial trimming); set to 0.8 to follow the text
# literally as a sensitivity check.
INIT_FRAC = 0.0


def _q_r(Xadj, labels):
    """Q_j^R = 2 x BCSS on the adjusted data X - E."""
    return 2.0 * bcss_per_feature(Xadj, labels)


def arsk_once(X, k, lam1, lam2, seed, max_iter=50, tol=1e-4, init_frac=INIT_FRAC):
    X = np.asarray(X, float)
    n, p = X.shape
    w = np.ones(p) / np.sqrt(p)

    # Initial centers and initial E (Alg. 1, step 2)
    centers, _ = kmeans_plusplus(X, n_clusters=k, random_state=seed)
    dist = ((X[:, None, :] - centers[None, :, :]) ** 2).sum(axis=2).min(axis=1)
    E = np.zeros_like(X)
    n_far = int(round(init_frac * n))
    if n_far > 0:
        far = np.argsort(dist)[::-1][:n_far]
        E[far] = X[far]

    labels = None
    converged = False
    for it in range(1, max_iter + 1):
        sw = np.sqrt(np.maximum(w, 0.0))
        Xs = (X - E) * sw                                   # step 3a
        if labels is None:
            init_c = centers * sw
        else:
            init_c = np.array([Xs[labels == c].mean(axis=0) if (labels == c).any()
                               else Xs[np.random.default_rng(seed + it).integers(n)]
                               for c in range(k)])
        labels, _ = weighted_lloyd(Xs, k, np.ones(p), init_c)

        # step 3b: E on the weighted (un-adjusted) data, cluster means
        Xw = X * sw
        mu = np.array([Xw[labels == c].mean(axis=0) if (labels == c).any()
                       else Xw.mean(axis=0) for c in range(k)])
        R = Xw - mu[labels]
        nrm = np.linalg.norm(R, axis=1, keepdims=True)
        Ew = R * np.maximum(0.0, 1.0 - lam1 / np.maximum(nrm, EPS_NUM))

        # step 3c: restore unweighted E (features with w_j = 0 carry no E)
        E = np.where(sw[None, :] > EPS_NUM, Ew / np.maximum(sw[None, :], EPS_NUM), 0.0)

        # step 3d: weights
        q = _q_r(X - E, labels)
        a = np.maximum(q - lam2, 0.0)
        w_old = w
        w = a / np.linalg.norm(a) if np.linalg.norm(a) > EPS_NUM else np.ones(p) / np.sqrt(p)

        # step 3e
        if np.abs(w - w_old).sum() / max(np.abs(w_old).sum(), EPS_NUM) < tol:
            converged = True
            break

    D = float((w * _q_r(X - E, labels)).sum())
    return {"labels": labels, "weights": w, "E": E, "D": max(D, EPS_NUM),
            "n_outlying": int((np.linalg.norm(E, axis=1) > 1e-9).sum()),
            "iterations": it, "converged": converged}


def _arsk_grids(X, k, seed, n_values=10):
    from sklearn.cluster import KMeans
    lab = KMeans(k, n_init=20, random_state=seed).fit_predict(X)
    q = _q_r(X, lab)
    mu = np.array([X[lab == c].mean(axis=0) for c in range(k)])
    rn = np.linalg.norm(X - mu[lab], axis=1)
    lam2_grid = q.max() * np.exp(np.linspace(np.log(0.9), np.log(0.01), n_values))
    lam1_grid = np.exp(np.linspace(np.log(rn.max()), np.log(np.percentile(rn, 20)), n_values))
    return lam1_grid, lam2_grid, rn.max()


def _robust_gap(X, Xperms, k, lam1, lam2, seed):
    D = arsk_once(X, k, lam1, lam2, seed)["D"]
    Db = [arsk_once(Xb, k, lam1, lam2, seed + 1 + b)["D"] for b, Xb in enumerate(Xperms)]
    return np.log(D) - np.mean(np.log(Db))


def select_arsk_lambdas(X, k, B=25, seed=2026, n_values=10):
    """Algorithm 2: lam2 first (lam1 = lam1_dagger), then lam1."""
    X = np.asarray(X, float)
    rng = np.random.default_rng(seed)
    Xperms = [np.column_stack([rng.permutation(X[:, j]) for j in range(X.shape[1])])
              for _ in range(B)]
    lam1_grid, lam2_grid, lam1_dagger = _arsk_grids(X, k, seed, n_values)

    gaps2 = [_robust_gap(X, Xperms, k, lam1_dagger, l2, seed) for l2 in lam2_grid]
    lam2 = float(lam2_grid[int(np.argmax(gaps2))])

    gaps1 = [_robust_gap(X, Xperms, k, l1, lam2, seed) for l1 in lam1_grid]
    lam1 = float(lam1_grid[int(np.argmax(gaps1))])
    return lam1, lam2, {"lam1_grid": lam1_grid, "gap_lam1": gaps1,
                        "lam2_grid": lam2_grid, "gap_lam2": gaps2}


def make_arsk_wrapper(X, tuning_seed=2026, B=25, verbose=True):
    """
    Returns fn(X, k, seed) for fit_across_seeds.
    (lam1, lam2) selected once per k by the robust Gap statistic and
    cached in fn.lambdas_by_k.
    """
    X = np.asarray(X, float)
    cache = {}

    def fn(X_in, k, seed):
        if k not in cache:
            l1, l2, _ = select_arsk_lambdas(X, k, B=B, seed=tuning_seed + 10 * k)
            cache[k] = (l1, l2)
            if verbose:
                print(f"    ARSK: k={k}, lambda1={l1:.4f}, lambda2={l2:.4f}")
        l1, l2 = cache[k]
        return arsk_once(X_in, k, l1, l2, seed)["labels"]

    fn.lambdas_by_k = cache
    return fn

In [16]:
#@title DBSCAN

def fit_dbscan_grid(X, eps_values=DBSCAN_EPS_VALUES, minpts_values=DBSCAN_MINPTS_VALUES,
                     min_size=MIN_CLUSTER_SIZE, min_retention=DBSCAN_MIN_RETENTION):
    n = X.shape[0]
    runs = []
    for eps in eps_values:
        for minpts in minpts_values:
            labels = DBSCAN(eps=eps, min_samples=minpts).fit_predict(X)
            n_clusters = len(set(labels) - {-1})
            retention = (labels != -1).sum() / n
            sil, dbi = safe_metrics(X, labels, ignore_noise=True)
            admissible = (n_clusters >= 2 and retention >= min_retention
                          and min_cluster_size(labels, ignore_noise=True) >= min_size
                          and not np.isnan(sil))
            runs.append({"eps": eps, "minpts": minpts, "labels": labels,
                         "n_clusters": n_clusters, "retention": retention,
                         "silhouette": sil, "dbi": dbi, "admissible": admissible})
    runs_df = pd.DataFrame(runs)
    adm = runs_df[runs_df.admissible]
    if len(adm) == 0:
        return None, runs_df
    return runs_df.loc[adm["silhouette"].idxmax()], runs_df

# Consistency across Algorithms

In [17]:
# @title Selected Parameter
all_selected = {}          # [weight_class][method] -> selected run (Series/dict)
all_agreement = []
all_coverage_adjusted = []
tuned_parameters = {}      # [weight_class] -> selected s / lambdas per k (report these)

for class_index, wc in enumerate(raw_data):
    X_p0 = prepared[wc]["P0"]["X"]
    methods = {}

    methods["Ward"], _ = fit_ward_all_k(X_p0)

    methods["k-Means"], _ = fit_across_seeds(X_p0, kmeans_once_wrapper, KMEANS_SEEDS)

    rskc_fn = make_rskc_wrapper(X_p0, tuning_seed=202600 + 1000 * class_index)
    methods["RSKC-STOF"], _ = fit_across_seeds(X_p0, rskc_fn, RSKC_SEEDS)

    arsk_fn = make_arsk_wrapper(X_p0, tuning_seed=2026)
    methods["ARSK"], _ = fit_across_seeds(X_p0, arsk_fn, ARSK_SEEDS)

    tuned_parameters[wc] = {"RSKC-STOF s": dict(rskc_fn.s_by_k),
                            "ARSK (lambda1, lambda2)": dict(arsk_fn.lambdas_by_k)}

    methods["DBSCAN"], _ = fit_dbscan_grid(X_p0)

    all_selected[wc] = methods

    print(f"\n=== {wc} (P0) ===")
    for name, sel in methods.items():
        if sel is None:
            print(f"  {name}: no admissible solution")
            continue
        k_display = sel["k"] if "k" in sel else sel.get("n_clusters")
        print(f"  {name}: k={k_display}  sil={sel['silhouette']:.3f}  dbi={sel['dbi']:.3f}")

    RSKC-STOF: k=2, selected s=1.6272
    RSKC-STOF: k=3, selected s=2.0448
    RSKC-STOF: k=4, selected s=2.2066
    RSKC-STOF: k=5, selected s=2.2066
    ARSK: k=2, lambda1=1.3450, lambda2=56.8450
    ARSK: k=3, lambda1=1.5161, lambda2=61.7116
    ARSK: k=4, lambda1=0.9291, lambda2=41.4876
    ARSK: k=5, lambda1=2.0596, lambda2=41.4876

=== Female-42kg (P0) ===
  Ward: k=4  sil=0.442  dbi=0.872
  k-Means: k=4  sil=0.437  dbi=0.931
  RSKC-STOF: k=4  sil=0.442  dbi=0.872
  ARSK: k=4  sil=0.439  dbi=0.899
  DBSCAN: k=3  sil=0.477  dbi=0.807
    RSKC-STOF: k=2, selected s=2.2066
    RSKC-STOF: k=3, selected s=2.0448
    RSKC-STOF: k=4, selected s=2.3812
    RSKC-STOF: k=5, selected s=2.2066
    ARSK: k=2, lambda1=1.0292, lambda2=51.5144
    ARSK: k=3, lambda1=3.1099, lambda2=62.7490
    ARSK: k=4, lambda1=1.0126, lambda2=41.0492
    ARSK: k=5, lambda1=1.3166, lambda2=40.6108

=== Female-47kg (P0) ===
  Ward: k=3  sil=0.307  dbi=1.277
  k-Means: k=5  sil=0.356  dbi=1.030
  RSKC-STOF: k=5 

In [18]:
# @title Table 4. Clustering Results of Five Algorithms on P0.
comparative_df     = pd.read_csv(OUTPUT_DIR / "table3_p0_clustering_results.csv")
comparative_column = ["Weight class", "Method", "Selected setting", "Sil.", "W/B", "Retained n (%)", "DBI", "Cluster sizes"]

comparative_df[comparative_column]

,Weight class,Method,Selected setting,Sil.,W/B,Retained n (%),DBI,Cluster sizes
0,Female-42kg,Ward,k=4,0.442,0.510,38 (100.0),0.872,"14, 5, 4, 15"
1,Female-42kg,k-Means,k=4,0.437,0.506,38 (100.0),0.931,"6, 15, 4, 13"
2,Female-42kg,RSKC-STOF,"k=4, s=2.21",0.442,0.510,38 (100.0),0.872,"15, 4, 5, 14"
3,Female-42kg,ARSK,"k=4, λ1=0.93, λ2=41.49",0.439,0.513,38 (100.0),0.899,"5, 13, 16, 4"
4,Female-42kg,DBSCAN,"ε=2.0, M=4",0.477,0.530,33 (86.8),0.807,"13, 5, 15"
5,Female-47kg,Ward,k=3,0.307,1.274,38 (100.0),1.277,"9, 7, 22"
6,Female-47kg,k-Means,k=5,0.356,0.501,38 (100.0),1.030,"10, 8, 7, 9, 4"
7,Female-47kg,RSKC-STOF,"k=5, s=2.21",0.319,0.670,38 (100.0),1.270,"9, 8, 9, 8, 4"
8,Female-47kg,ARSK,"k=5, λ1=1.32, λ2=40.61",0.319,0.670,38 (100.0),1.270,"8, 8, 9, 4, 9"
9,Female-47kg,DBSCAN,"ε=2.4, M=4",0.364,1.714,31 (81.6),0.882,"9, 22"


In [19]:
def _ari_nmi(labels_a, labels_b):
    """ARI and NMI; NaN if fewer than 2 observations."""
    if len(labels_a) < 2:
        return np.nan, np.nan
    return (
        adjusted_rand_score(labels_a, labels_b),
        normalized_mutual_info_score(labels_a, labels_b, average_method="arithmetic"),
    )


def to_singletons(labels):
    """
    Relabel each DBSCAN noise point (-1) as its own one-member cluster,
    so noise is kept in the comparison but never treated as a coherent group.
    """
    labels = np.asarray(labels).astype(int).copy()
    noise = labels == -1
    if noise.any():
        start = labels.max() + 1          # works even if all points are noise
        labels[noise] = start + np.arange(noise.sum())
    return labels


def agreement_three_ways(labels_a, labels_b):
    """
    1. Conditional       : ARI/NMI on observations that are non-noise in BOTH methods
                           (original analysis).
    2. Coverage-adjusted : conditional ARI/NMI x coverage, where
                           coverage = jointly retained n / full n.
                           Noise points contribute zero agreement.
    3. Noise-retained    : ARI/NMI on the FULL sample, with each noise point
                           treated as its own singleton cluster.

    For pairs without noise, all three are identical to the full-sample value.
    """
    a = np.asarray(labels_a).astype(int)
    b = np.asarray(labels_b).astype(int)

    if len(a) != len(b):
        raise ValueError("Label vectors must be aligned to the same observations.")

    n = len(a)
    retained = (a != -1) & (b != -1)
    n_retained = int(retained.sum())
    coverage = n_retained / n

    # 1. Conditional
    ari_c, nmi_c = _ari_nmi(a[retained], b[retained])

    # 2. Coverage-adjusted
    ari_cov = ari_c * coverage if np.isfinite(ari_c) else np.nan
    nmi_cov = nmi_c * coverage if np.isfinite(nmi_c) else np.nan

    # 3. Noise-retained (singletons)
    ari_s, nmi_s = _ari_nmi(to_singletons(a), to_singletons(b))

    return {
        "n_full": n,
        "n_retained": n_retained,
        "n_noise": n - n_retained,
        "coverage": coverage,
        "ARI_conditional": ari_c,
        "NMI_conditional": nmi_c,
        "ARI_coverage_adj": ari_cov,
        "NMI_coverage_adj": nmi_cov,
        "ARI_noise_singleton": ari_s,
        "NMI_noise_singleton": nmi_s,
    }

In [20]:
all_agreement = []

for wc in raw_data:
    methods = all_selected[wc]
    names = list(methods.keys())

    for i in range(len(names)):
        for j in range(i + 1, len(names)):
            a, b = methods[names[i]], methods[names[j]]
            if a is None or b is None:
                continue

            res = agreement_three_ways(a["labels"], b["labels"])

            all_agreement.append({
                "weight_class": wc,
                "pair": f"{names[i]}-{names[j]}",
                "involves_noise": res["n_noise"] > 0,
                **res,
            })

agreement_df = pd.DataFrame(all_agreement)


# ============================================================
# SANITY CHECK: pairs without noise must give identical values
# ============================================================

no_noise = agreement_df[~agreement_df["involves_noise"]]
for metric in ["ARI", "NMI"]:
    assert np.allclose(no_noise[f"{metric}_conditional"], no_noise[f"{metric}_coverage_adj"])
    assert np.allclose(no_noise[f"{metric}_conditional"], no_noise[f"{metric}_noise_singleton"])

In [21]:
# @title Table 5. Partition Agreement across Algorithms.
coverage_adjusted_df = agreement_df[agreement_df["involves_noise"]].reset_index(drop=True)

table5_cols = [
    "weight_class", "pair", "n_retained", "coverage",
    "ARI_conditional", "ARI_coverage_adj", "ARI_noise_singleton",
    "NMI_conditional", "NMI_coverage_adj", "NMI_noise_singleton",
]

table5 = agreement_df[table5_cols].round(3)
display(table5)

,weight_class,pair,n_retained,coverage,ARI_conditional,ARI_coverage_adj,ARI_noise_singleton,NMI_conditional,NMI_coverage_adj,NMI_noise_singleton
0,Female-42kg,Ward-k-Means,38,1.000,0.939,0.939,0.939,0.934,0.934,0.934
1,Female-42kg,Ward-RSKC-STOF,38,1.000,1.000,1.000,1.000,1.000,1.000,1.000
2,Female-42kg,Ward-ARSK,38,1.000,0.906,0.906,0.906,0.922,0.922,0.922
3,Female-42kg,Ward-DBSCAN,33,0.868,1.000,0.868,0.934,1.000,0.868,0.911
4,Female-42kg,k-Means-RSKC-STOF,38,1.000,0.939,0.939,0.939,0.934,0.934,0.934
5,Female-42kg,k-Means-ARSK,38,1.000,0.932,0.932,0.932,0.932,0.932,0.932
6,Female-42kg,k-Means-DBSCAN,33,0.868,1.000,0.868,0.961,1.000,0.868,0.921
7,Female-42kg,RSKC-STOF-ARSK,38,1.000,0.906,0.906,0.906,0.922,0.922,0.922
8,Female-42kg,RSKC-STOF-DBSCAN,33,0.868,1.000,0.868,0.934,1.000,0.868,0.911
9,Female-42kg,ARSK-DBSCAN,33,0.868,1.000,0.868,0.927,1.000,0.868,0.910


# Outcome Independence of the Clusters

In [22]:
# @title Table 6. Clusters under Match-Context and Outcome Features (S1, S2).
comparative_S1S2  = pd.read_csv(OUTPUT_DIR / 'table7_feature_sensitivity.csv')
comparative_S1S2

,Weight class,Method,S1 k,ΔSil. S1,ARI S1/P0,ARI S1/P0 (noise-retained),S2 k,ΔSil. S2,ARI S2/P0,ARI S2/P0 (noise-retained)
0,Female-42kg,Ward,4.0,-0.120,1.000,1.000,2.0,-0.118,0.470,0.470
1,Female-42kg,k-Means,4.0,-0.117,0.932,0.932,4.0,-0.107,1.000,1.000
2,Female-42kg,RSKC-STOF,3.0,-0.156,0.770,0.770,2.0,-0.135,0.567,0.567
3,Female-42kg,ARSK,3.0,-0.154,0.684,0.684,5.0,-0.131,0.729,0.729
4,Female-42kg,DBSCAN,2.0,-0.206,0.344,0.470,3.0,-0.078,1.000,0.915
5,Female-47kg,Ward,4.0,-0.025,0.227,0.227,5.0,-0.009,0.486,0.486
6,Female-47kg,k-Means,5.0,-0.071,0.480,0.480,5.0,-0.054,0.450,0.450
7,Female-47kg,RSKC-STOF,5.0,-0.070,0.585,0.585,5.0,-0.041,0.578,0.578
8,Female-47kg,ARSK,5.0,-0.080,0.303,0.303,5.0,-0.036,0.395,0.395
9,Female-47kg,DBSCAN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [23]:
# @title Run all methods on TP, NT and SO
SENS_CONFIGS = ["TP", "NT", "SO"]

sens_selected = {cfg: {} for cfg in SENS_CONFIGS}   # [cfg][wc][method] -> selected run
sens_tuned    = {cfg: {} for cfg in SENS_CONFIGS}   # [cfg][wc]         -> s / lambdas per k

for cfg in SENS_CONFIGS:
    for class_index, wc in enumerate(raw_data):
        X = prepared[wc][cfg]["X"]
        rskc_fn = make_rskc_wrapper(X, tuning_seed=202600 + 1000 * class_index, verbose=False)
        arsk_fn = make_arsk_wrapper(X, tuning_seed=2026, verbose=False)

        fitters = {"Ward":      lambda: fit_ward_all_k(X),
                   "k-Means":   lambda: fit_across_seeds(X, kmeans_once_wrapper, KMEANS_SEEDS),
                   "RSKC-STOF": lambda: fit_across_seeds(X, rskc_fn, RSKC_SEEDS),
                   "ARSK":      lambda: fit_across_seeds(X, arsk_fn, ARSK_SEEDS),
                   "DBSCAN":    lambda: fit_dbscan_grid(X)}
        m = {}
        for name, fit in fitters.items():
          m[name], _ = fit()
        sens_selected[cfg][wc] = m
        sens_tuned[cfg][wc] = {"RSKC-STOF s": dict(rskc_fn.s_by_k),
                               "ARSK (lambda1, lambda2)": dict(arsk_fn.lambdas_by_k)}

        print(f"\n=== {wc} ({cfg}, n={X.shape[0]}, features={prepared[wc][cfg]['features']}) ===")
        for name, sel in m.items():
            if sel is None:
                print(f"  {name}: no admissible solution")
                continue
            k_display = sel["k"] if "k" in sel else sel.get("n_clusters")
            print(f"  {name}: k={k_display}  sil={sel['silhouette']:.3f}  dbi={sel['dbi']:.3f}")


=== Female-42kg (TP, n=38, features=['yuko', 'waza_ari', 'ippon', 'attack', 'defend', 'timing_event', 'longest_point_in_sec_cond', 'fastest_point_in_sec_cond']) ===
  Ward: k=4  sil=0.339  dbi=1.138
  k-Means: k=4  sil=0.339  dbi=1.168
  RSKC-STOF: k=4  sil=0.334  dbi=1.143
  ARSK: k=4  sil=0.291  dbi=1.189
  DBSCAN: k=2  sil=0.339  dbi=0.974

=== Female-47kg (TP, n=38, features=['yuko', 'waza_ari', 'ippon', 'attack', 'defend', 'timing_event', 'longest_point_in_sec_cond', 'fastest_point_in_sec_cond']) ===
  Ward: k=3  sil=0.303  dbi=1.343
  k-Means: k=5  sil=0.360  dbi=1.057
  RSKC-STOF: k=2  sil=0.336  dbi=1.455
  ARSK: k=5  sil=0.329  dbi=1.082
  DBSCAN: no admissible solution

=== Female-54kg (TP, n=38, features=['yuko', 'attack', 'defend', 'timing_event', 'longest_point_in_sec_cond', 'fastest_point_in_sec_cond']) ===
  Ward: k=3  sil=0.393  dbi=1.041
  k-Means: k=3  sil=0.393  dbi=1.041
  RSKC-STOF: k=3  sil=0.393  dbi=1.041
  ARSK: k=4  sil=0.347  dbi=1.048
  DBSCAN: k=3  sil=0.4

In [25]:
# @title Table 7.a Homogeneity(Sc., P0) and ARI(Sc., P0)
from sklearn.metrics import homogeneity_score, adjusted_rand_score

rows = []
for wc in prepared.keys():
    scored = np.asarray(prepared[wc]["SO"]["rows"])      # True = scorer
    for method, sel in all_selected[wc].items():
        if sel is None:                                   # no admissible solution
            continue
        labels = np.asarray(sel["labels"])
        keep = labels != -1                               # DBSCAN noise = -1

        rows.append({
            "class_weight": wc,
            "method"      : method,
            "k"           : len(np.unique(labels[keep])),
            "n_r"         : int(keep.sum()),

            # noise excluded (same as conditional ARI)
            "H"           : homogeneity_score(scored[keep], labels[keep]),
            "ARI"         : adjusted_rand_score(scored[keep], labels[keep]),

            # noise kept as its own group (only differs for DBSCAN)
            "H_noise"     : homogeneity_score(scored, to_singletons(labels)),
            "ARI_noise"   : adjusted_rand_score(scored, to_singletons(labels)),
        })

homogeneity_df = pd.DataFrame(rows).round(3)
homogeneity_df.head()

,class_weight,method,k,n_r,H,ARI,H_noise,ARI_noise
0,Female-42kg,Ward,4,38,1.000,0.567,1.000,0.567
1,Female-42kg,k-Means,4,38,0.892,0.517,0.892,0.517
2,Female-42kg,RSKC-STOF,4,38,1.000,0.567,1.000,0.567
3,Female-42kg,ARSK,4,38,0.850,0.488,0.850,0.488
4,Female-42kg,DBSCAN,3,33,1.000,0.717,1.000,0.515


In [27]:
homogeneity_df[homogeneity_df.class_weight == 'Female-42kg']

,class_weight,method,k,n_r,H,ARI,H_noise,ARI_noise
0,Female-42kg,Ward,4,38,1.000,0.567,1.000,0.567
1,Female-42kg,k-Means,4,38,0.892,0.517,0.892,0.517
2,Female-42kg,RSKC-STOF,4,38,1.000,0.567,1.000,0.567
3,Female-42kg,ARSK,4,38,0.850,0.488,0.850,0.488
4,Female-42kg,DBSCAN,3,33,1.000,0.717,1.000,0.515


In [29]:
# @title Table 7.a Homogeneity(Sc., TP) and ARI (Sc., TP).
rows = []
for wc in prepared.keys():
    scored = np.asarray(prepared[wc]["SO"]["rows"])      # True = scorer
    for method, sel in sens_selected['TP'][wc].items():
        if sel is None:                                   # no admissible solution
            continue
        labels = np.asarray(sel["labels"])
        keep = labels != -1                               # DBSCAN noise = -1

        rows.append({
            "class_weight": wc,
            "method"      : method,
            "k"           : len(np.unique(labels[keep])),
            "n_r"         : int(keep.sum()),

            # noise excluded (same as conditional ARI)
            "H"           : homogeneity_score(scored[keep], labels[keep]),
            "ARI"         : adjusted_rand_score(scored[keep], labels[keep]),

            # noise kept as its own group (only differs for DBSCAN)
            "H_noise"     : homogeneity_score(scored, to_singletons(labels)),
            "ARI_noise"   : adjusted_rand_score(scored, to_singletons(labels)),
        })

homogeneity_TP_df = pd.DataFrame(rows).round(3)
homogeneity_TP_df.head()

,class_weight,method,k,n_r,H,ARI,H_noise,ARI_noise
0,Female-42kg,Ward,4,38,0.850,0.488,0.850,0.488
1,Female-42kg,k-Means,4,38,0.892,0.517,0.892,0.517
2,Female-42kg,RSKC-STOF,4,38,1.000,0.567,1.000,0.567
3,Female-42kg,ARSK,4,38,0.802,0.541,0.802,0.541
4,Female-42kg,DBSCAN,2,31,0.113,-0.020,0.252,-0.065


In [31]:
# @title Table 7.a Homogeneity(Sc., NT) and ARI(Sc., NT).
rows = []
for wc in prepared.keys():
    scored = np.asarray(prepared[wc]["SO"]["rows"])      # True = scorer
    for method, sel in sens_selected['NT'][wc].items():
        if sel is None:                                   # no admissible solution
            continue
        labels = np.asarray(sel["labels"])
        keep = labels != -1                               # DBSCAN noise = -1

        rows.append({
            "class_weight": wc,
            "method"      : method,
            "k"           : len(np.unique(labels[keep])),
            "n_r"         : int(keep.sum()),

            # noise excluded (same as conditional ARI)
            "H"           : homogeneity_score(scored[keep], labels[keep]),
            "ARI"         : adjusted_rand_score(scored[keep], labels[keep]),

            # noise kept as its own group (only differs for DBSCAN)
            "H_noise"     : homogeneity_score(scored, to_singletons(labels)),
            "ARI_noise"   : adjusted_rand_score(scored, to_singletons(labels)),
        })

homogeneity_NT_df = pd.DataFrame(rows).round(3)
homogeneity_NT_df.head()

,class_weight,method,k,n_r,H,ARI,H_noise,ARI_noise
0,Female-42kg,Ward,4,38,0.333,0.096,0.333,0.096
1,Female-42kg,k-Means,4,38,0.441,0.180,0.441,0.180
2,Female-42kg,RSKC-STOF,4,38,0.396,0.134,0.396,0.134
3,Female-42kg,ARSK,4,38,0.308,0.093,0.308,0.093
4,Female-42kg,DBSCAN,2,32,0.105,-0.031,0.227,-0.066


In [30]:
homogeneity_TP_df[homogeneity_TP_df.class_weight == 'Female-42kg']

,class_weight,method,k,n_r,H,ARI,H_noise,ARI_noise
0,Female-42kg,Ward,4,38,0.850,0.488,0.850,0.488
1,Female-42kg,k-Means,4,38,0.892,0.517,0.892,0.517
2,Female-42kg,RSKC-STOF,4,38,1.000,0.567,1.000,0.567
3,Female-42kg,ARSK,4,38,0.802,0.541,0.802,0.541
4,Female-42kg,DBSCAN,2,31,0.113,-0.020,0.252,-0.065


In [32]:
# @title Build the structural-zero table
# DBSCAN noise is handled exactly as in Table 5 (agreement_three_ways):
#   conditional     : only observations that are non-noise in BOTH partitions
#   coverage-adj.   : conditional x (jointly retained n / full n)
#   noise-retained  : full sample, every noise point its own singleton cluster
# For methods without noise the three values are identical.
# The manuscript table reports the NOISE-RETAINED value (full sample, same
# basis for every method); all three are kept in the long CSV.

def _k(sel):
    return int(sel["k"] if "k" in sel else sel["n_clusters"])

def _agree(a, b):
    r = agreement_three_ways(a, b)
    return {"cond": r["ARI_conditional"],
            "cov": r["ARI_coverage_adj"],
            "ret": r["ARI_noise_singleton"],
            "coverage": r["coverage"]}

long_rows = []
for wc in raw_data:
    #labeled on score or not score
    scored = (raw_data[wc][TIMING_FEATURES] > 0).any(axis=1).to_numpy().astype(int)

    for name in ["Ward", "k-Means", "RSKC-STOF", "ARSK", "DBSCAN"]:
        p0 = all_selected[wc][name]
        if p0 is None:
            continue
        lab_p0 = np.asarray(p0["labels"]).astype(int)
        s = _agree(scored, lab_p0) #compare to score or not score

        long_rows.append({
            "weight_class": wc,
            "method"      : name,
            "config"      : "P0",
            "n"           : len(lab_p0),
            "retained_n"  : int((lab_p0 != -1).sum()),
            "k"           : _k(p0),
            "silhouette"  : p0["silhouette"],
            "dbi"         : p0["dbi"],
            "delta_sil"   : 0.0,

            # reference row: comparing P0 with itself is not informative
            "ARI_vs_P0_cond" : np.nan, "ARI_vs_P0_cov": np.nan, "ARI_vs_P0_ret": np.nan,

            "ARI_scored_cond": s["cond"], "ARI_scored_cov": s["cov"], "ARI_scored_ret": s["ret"],

            "coverage"       : s["coverage"],
        })

        for cfg in SENS_CONFIGS:
            sel  = sens_selected[cfg][wc][name]
            rows = prepared[wc][cfg]["rows"]
            if sel is None:
                long_rows.append({"weight_class": wc, "method": name, "config": cfg,
                                  "n": int(rows.sum()), "k": np.nan})
                continue
            lab = np.asarray(sel["labels"]).astype(int) #label for TP, NT, SO

            p = _agree(lab_p0[rows], lab)

            #if not SO
            s = _agree(scored[rows], lab) if cfg != "SO" else {"cond": np.nan, "cov": np.nan, "ret": np.nan}

            long_rows.append({
                "weight_class": wc,
                "method": name,
                "config": cfg,
                "n": int(rows.sum()),
                "retained_n": int((lab != -1).sum()),
                "k": _k(sel),
                "silhouette": sel["silhouette"],
                "dbi": sel["dbi"],

                # computed from unrounded values (rounded only for display)
                "delta_sil": sel["silhouette"] - p0["silhouette"],
                "ARI_vs_P0_cond": p["cond"], "ARI_vs_P0_cov": p["cov"], "ARI_vs_P0_ret": p["ret"],

                # undefined for SO: every observation there scored
                "ARI_scored_cond": s["cond"], "ARI_scored_cov": s["cov"], "ARI_scored_ret": s["ret"],

                "coverage": p["coverage"],
            })

zero_long = pd.DataFrame(long_rows)

In [35]:
zero_long[zero_long.weight_class == 'Female-42kg'].head(8)

,weight_class,method,config,n,retained_n,k,silhouette,dbi,delta_sil,ARI_vs_P0_cond,ARI_vs_P0_cov,ARI_vs_P0_ret,ARI_scored_cond,ARI_scored_cov,ARI_scored_ret,coverage
0,Female-42kg,Ward,P0,38,38.0,4.0,0.441577,0.871717,0.000000,NaN,NaN,NaN,0.566617,0.566617,0.566617,1.0
1,Female-42kg,Ward,TP,38,38.0,4.0,0.339192,1.138046,-0.102385,0.905702,0.905702,0.905702,0.488203,0.488203,0.488203,1.0
2,Female-42kg,Ward,NT,38,38.0,4.0,0.352427,0.980733,-0.089150,0.441673,0.441673,0.441673,0.095548,0.095548,0.095548,1.0
3,Female-42kg,Ward,SO,24,24.0,2.0,0.242866,1.286012,-0.198712,0.581290,0.581290,0.581290,NaN,NaN,NaN,1.0
4,Female-42kg,k-Means,P0,38,38.0,4.0,0.437444,0.930939,0.000000,NaN,NaN,NaN,0.516766,0.516766,0.516766,1.0
5,Female-42kg,k-Means,TP,38,38.0,4.0,0.338693,1.168167,-0.098752,1.000000,1.000000,1.000000,0.516766,0.516766,0.516766,1.0
6,Female-42kg,k-Means,NT,38,38.0,4.0,0.363527,0.923268,-0.073917,0.563754,0.563754,0.563754,0.179553,0.179553,0.179553,1.0
7,Female-42kg,k-Means,SO,24,24.0,3.0,0.238498,1.370928,-0.198946,0.859047,0.859047,0.859047,NaN,NaN,NaN,1.0


In [36]:
def _get(wc, name, cfg, col):
    v = zero_long.query("weight_class == @wc and method == @name and config == @cfg")[col]
    return v.iloc[0] if len(v) else np.nan

In [37]:
# ---- compact manuscript table (noise-retained ARI) ----

table_sz = []
for wc in raw_data:
    for name in ["Ward", "k-Means", "RSKC-STOF", "ARSK", "DBSCAN"]:
        if all_selected[wc][name] is None:
            continue
        table_sz.append({
            "Weight class": wc, "Method": name,
            "P0 k": _get(wc, name, "P0", "k"),
            "P0 retained n": _get(wc, name, "P0", "retained_n"),
            "P0 ARI scored": _get(wc, name, "P0", "ARI_scored_ret"),
            "TP k": _get(wc, name, "TP", "k"),
            "TP ΔSil": _get(wc, name, "TP", "delta_sil"),
            "TP ARI/P0": _get(wc, name, "TP", "ARI_vs_P0_ret"),
            "TP ARI scored": _get(wc, name, "TP", "ARI_scored_ret"),
            "NT k": _get(wc, name, "NT", "k"),
            "NT ΔSil": _get(wc, name, "NT", "delta_sil"),
            "NT ARI/P0": _get(wc, name, "NT", "ARI_vs_P0_ret"),
            "SO k": _get(wc, name, "SO", "k"),
            "SO ARI/P0": _get(wc, name, "SO", "ARI_vs_P0_ret"),
        })
table_sz = pd.DataFrame(table_sz).round(3)
display(table_sz)

,Weight class,Method,P0 k,P0 retained n,P0 ARI scored,TP k,TP ΔSil,TP ARI/P0,TP ARI scored,NT k,NT ΔSil,NT ARI/P0,SO k,SO ARI/P0
0,Female-42kg,Ward,4.0,38.0,0.567,4.0,-0.102,0.906,0.488,4.0,-0.089,0.442,2.0,0.581
1,Female-42kg,k-Means,4.0,38.0,0.517,4.0,-0.099,1.000,0.517,4.0,-0.074,0.564,3.0,0.859
2,Female-42kg,RSKC-STOF,4.0,38.0,0.567,4.0,-0.108,1.000,0.567,4.0,-0.124,0.494,2.0,0.219
3,Female-42kg,ARSK,4.0,38.0,0.488,4.0,-0.149,0.663,0.541,4.0,-0.127,0.276,5.0,0.134
4,Female-42kg,DBSCAN,3.0,33.0,0.515,2.0,-0.138,0.435,-0.065,2.0,-0.045,0.457,2.0,0.654
5,Female-47kg,Ward,3.0,38.0,0.581,3.0,-0.003,0.766,0.496,2.0,-0.089,0.022,2.0,1.000
6,Female-47kg,k-Means,5.0,38.0,0.248,5.0,0.004,1.000,0.248,5.0,-0.024,0.437,2.0,0.348
7,Female-47kg,RSKC-STOF,5.0,38.0,0.244,2.0,0.017,0.008,-0.106,3.0,-0.051,0.058,2.0,-0.031
8,Female-47kg,ARSK,5.0,38.0,0.244,5.0,0.010,0.543,0.264,5.0,-0.021,0.408,2.0,-0.031
9,Female-47kg,DBSCAN,2.0,31.0,0.531,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [38]:
# ---- DBSCAN rows: all three agreement versions side by side ----
dbscan_detail = (zero_long[zero_long["method"] == "DBSCAN"]
                 [["weight_class", "config", "retained_n", "coverage",
                   "ARI_vs_P0_cond", "ARI_vs_P0_cov", "ARI_vs_P0_ret",
                   "ARI_scored_cond", "ARI_scored_cov", "ARI_scored_ret"]]
                 .round(3))

# ---- sanity check: methods without noise -> three versions identical ----
no_noise = zero_long[(zero_long["method"] != "DBSCAN") & zero_long["k"].notna()]
assert np.allclose(no_noise["ARI_vs_P0_cond"], no_noise["ARI_vs_P0_ret"], equal_nan=True)

print("\nDBSCAN agreement (conditional / coverage-adjusted / noise-retained):")
display(dbscan_detail)


DBSCAN agreement (conditional / coverage-adjusted / noise-retained):


,weight_class,config,retained_n,coverage,ARI_vs_P0_cond,ARI_vs_P0_cov,ARI_vs_P0_ret,ARI_scored_cond,ARI_scored_cov,ARI_scored_ret
16,Female-42kg,P0,33.0,0.868,NaN,NaN,NaN,0.717,0.623,0.515
17,Female-42kg,TP,31.0,0.816,0.309,0.252,0.435,-0.020,-0.016,-0.065
18,Female-42kg,NT,32.0,0.842,0.304,0.256,0.457,-0.031,-0.026,-0.066
19,Female-42kg,SO,20.0,0.792,0.777,0.615,0.654,NaN,NaN,NaN
36,Female-47kg,P0,31.0,0.816,NaN,NaN,NaN,1.000,0.816,0.531
37,Female-47kg,TP,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
38,Female-47kg,NT,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
39,Female-47kg,SO,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
56,Female-54kg,P0,31.0,0.816,NaN,NaN,NaN,1.000,0.816,0.657
57,Female-54kg,TP,32.0,0.763,0.577,0.440,0.465,0.646,0.544,0.407


In [39]:
# @title Table 7. Clusters under Alternative Representations of Structural Zeros

# @title Table 7: clusters under alternative representations of structural zeros
# Needs: raw_data, prepared, all_selected, sens_selected, TIMING_FEATURES,
#        to_singletons (from the agreement section).
# DBSCAN gets two rows:
#   DBSCAN1 = noise excluded (only observations non-noise in both partitions)
#   DBSCAN2 = noise retained, each noise point its own singleton cluster
from sklearn.metrics import adjusted_rand_score, homogeneity_score

METHOD_ORDER = ["Ward", "k-Means", "RSKC-STOF", "ARSK", "DBSCAN"]


def _labels(sel):
    return None if sel is None else np.asarray(sel["labels"]).astype(int)


def _n_clusters(lab, singletons):
    """k of a partition; with singletons, every noise point counts as a cluster."""
    k = len(set(lab.tolist()) - {-1})
    return k + int((lab == -1).sum()) if singletons else k


def _homog(sc, lab, singletons):
    """Homogeneity of the partition with respect to the scorer split (1 = every cluster pure)."""
    if singletons:
        return homogeneity_score(sc, to_singletons(lab))
    keep = lab != -1
    return homogeneity_score(sc[keep], lab[keep])


def _ari(a, b, singletons):
    """ARI between two aligned partitions (noise excluded, or noise as singletons)."""
    if singletons:
        return adjusted_rand_score(to_singletons(a), to_singletons(b))
    keep = (a != -1) & (b != -1)
    return adjusted_rand_score(a[keep], b[keep])


def _fmt(x, k=None):
    if x is None or (isinstance(x, float) and np.isnan(x)):
        return "–"
    s = f"{x:.3f}"
    return s if k is None else f"{s} ({k})"


t7_rows = []
for wc in raw_data:
    sc_all = (raw_data[wc][TIMING_FEATURES] > 0).any(axis=1).to_numpy().astype(int)

    for m in METHOD_ORDER:
        p0_sel = all_selected[wc][m]
        if p0_sel is None:
            continue
        p0 = _labels(p0_sel)
        variants = [(m, False)] if m != "DBSCAN" else [("DBSCAN1", False), ("DBSCAN2", True)]

        for row_name, sing in variants:
            row = {"Weight class": wc, "Method": row_name}

            # Homogeneity with the scorer split: P0, TP, NT (all on the full 38 rows)
            row["Homog. P0"] = _fmt(_homog(sc_all, p0, sing))
            for cfg in ["TP", "NT"]:
                lab = _labels(sens_selected[cfg][wc][m])
                row[f"Homog. {cfg}"] = "–" if lab is None else _fmt(_homog(sc_all, lab, sing))

            # ARI of the P0 partition with the scorer split (criterion iii)
            row["ARI Sc."] = _fmt(_ari(sc_all, p0, sing))

            # ARI of each alternative partition with P0 (on the rows that configuration uses)
            for cfg in ["TP", "NT", "SO"]:
                lab = _labels(sens_selected[cfg][wc][m])
                if lab is None:
                    row[f"ARI {cfg} (k)"] = "–"
                    continue
                rows = prepared[wc][cfg]["rows"]
                row[f"ARI {cfg} (k)"] = _fmt(_ari(p0[rows], lab, sing), _n_clusters(lab, sing))

            # Silhouette change (%) relative to P0; reported once per method
            for cfg in ["TP", "NT", "SO"]:
                sel = sens_selected[cfg][wc][m]
                if sing or sel is None:
                    row[f"ΔSil {cfg} (%)"] = "–"
                else:
                    d = 100 * (sel["silhouette"] - p0_sel["silhouette"]) / p0_sel["silhouette"]
                    row[f"ΔSil {cfg} (%)"] = f"{d:.1f}"

            t7_rows.append(row)

table7_zeros = pd.DataFrame(t7_rows)
#table7_zeros.to_csv(OUTPUT_DIR / "table7_structural_zeros.csv", index=False)
with pd.option_context("display.max_columns", None, "display.width", 250):
    display(table7_zeros)
print("DBSCAN1: noise excluded. DBSCAN2: noise retained as singleton clusters "
      "(k counts each noise point as a cluster). –: no admissible solution or not defined.")

,Weight class,Method,Homog. P0,Homog. TP,Homog. NT,ARI Sc.,ARI TP (k),ARI NT (k),ARI SO (k),ΔSil TP (%),ΔSil NT (%),ΔSil SO (%)
0,Female-42kg,Ward,1.000,0.850,0.333,0.567,0.906 (4),0.442 (4),0.581 (2),-23.2,-20.2,-45.0
1,Female-42kg,k-Means,0.892,0.892,0.441,0.517,1.000 (4),0.564 (4),0.859 (3),-22.6,-16.9,-45.5
2,Female-42kg,RSKC-STOF,1.000,1.000,0.396,0.567,1.000 (4),0.494 (4),0.219 (2),-24.5,-28.1,-49.7
3,Female-42kg,ARSK,0.850,0.802,0.308,0.488,0.663 (4),0.276 (4),0.134 (5),-33.9,-28.8,-50.5
4,Female-42kg,DBSCAN1,1.000,0.113,0.105,0.717,0.309 (2),0.304 (2),0.777 (2),-28.8,-9.3,-48.0
5,Female-42kg,DBSCAN2,1.000,0.252,0.227,0.515,0.435 (9),0.457 (8),0.654 (6),–,–,–
6,Female-47kg,Ward,1.000,1.000,0.092,0.581,0.766 (3),0.022 (2),1.000 (2),-1.1,-28.9,6.7
7,Female-47kg,k-Means,1.000,1.000,0.435,0.248,1.000 (5),0.437 (5),0.348 (2),1.0,-6.7,-1.6
8,Female-47kg,RSKC-STOF,1.000,0.119,0.215,0.244,0.008 (2),0.058 (3),-0.031 (2),5.2,-16.1,9.9
9,Female-47kg,ARSK,1.000,1.000,0.259,0.244,0.543 (5),0.408 (5),-0.031 (2),3.1,-6.5,9.9


DBSCAN1: noise excluded. DBSCAN2: noise retained as singleton clusters (k counts each noise point as a cluster). –: no admissible solution or not defined.


# Reproducibility under Data Perturbation

## Non Random Structure

In [ ]:
METHODS = ["Ward", "k-Means", "RSKC-STOF", "ARSK", "DBSCAN"]

In [ ]:
import pickle
ANALYSES       = ["loao", "subsample", "permutation", "bootstrap"]
CHECKPOINT_DIR = OUTPUT_DIR / "resampling_checkpoints"

In [ ]:
def load_replicates(analysis):
    recs = []
    for wc in raw_data:
        for p in sorted((CHECKPOINT_DIR / analysis / wc).glob("rep_*.pkl")):
            with open(p, "rb") as f:
                recs.append(pickle.load(f))
    return recs

replicates = {a: load_replicates(a) for a in ANALYSES}

In [ ]:
print(replicates.keys())
print(replicates['permutation'][0].keys())

dict_keys(['loao', 'subsample', 'permutation', 'bootstrap'])
dict_keys(['analysis', 'weight_class', 'rep', 'rows', 'left_out', 'reselected', 'fixed', 'timing'])


In [ ]:
# @title permutation null test (fixed vs full pipeline)
def _kdist(ks):
    ks = pd.Series(ks).dropna().astype(int)
    if not len(ks):
        return "—"
    vc = ks.value_counts(normalize=True).sort_index() * 100
    return ", ".join(f"{k}:{v:.0f}%" for k, v in vc.items())

null_test = []
for wc in raw_data:
    recs = [r for r in replicates["permutation"] if r["weight_class"] == wc]
    B = len(recs)
    for m in METHODS:
        p0 = all_selected[wc][m]
        if p0 is None:
            continue
        obs = float(p0["silhouette"])
        for mode, label in [("fixed", "Fixed (as published)"), ("reselected", "Full pipeline")]:
            vals = [r[mode][m] for r in recs]
            sils = np.array([v["silhouette"] for v in vals if v is not None and v["admissible"]])
            n_adm = len(sils)
            null_test.append({
                "Weight class": wc, "Method": m, "Null": label, "Observed sil.": obs,
                "B": B, "Admissible null": n_adm,
                "Null median": np.median(sils) if n_adm else np.nan,
                "Null 95th pct.": np.percentile(sils, 95) if n_adm else np.nan,
                "Δ Sil.": obs - np.median(sils) if n_adm else np.nan,
                # denominator = admissible null data sets (manuscript convention)
                "p (admissible)": (1 + np.sum(sils >= obs)) / (n_adm + 1) if n_adm else np.nan,
                # denominator = all B; no admissible solution = no structure found
                "p (all B)": (1 + np.sum(sils >= obs)) / (B + 1),
                "Null k distribution": _kdist([v["k"] for v in vals
                                               if v is not None and v["admissible"]])
                                       if mode == "reselected" else "",
            })
null_test_df = pd.DataFrame(null_test)
num = null_test_df.select_dtypes("number").columns
null_test_df[num] = null_test_df[num].round(4)
display(null_test_df)

,Weight class,Method,Null,Observed sil.,B,Admissible null,Null median,Null 95th pct.,Δ Sil.,p (admissible),p (all B),Null k distribution
0,Female-42kg,Ward,Fixed (as published),0.4416,500,404,0.1971,0.2423,0.2445,0.0025,0.0020,
1,Female-42kg,Ward,Full pipeline,0.4416,500,489,0.2755,0.3209,0.1660,0.0020,0.0020,"2:64%, 3:22%, 4:7%, 5:7%"
2,Female-42kg,k-Means,Fixed (as published),0.4374,500,449,0.2045,0.2438,0.2330,0.0022,0.0020,
3,Female-42kg,k-Means,Full pipeline,0.4374,500,500,0.2434,0.3210,0.1941,0.0020,0.0020,"2:38%, 3:15%, 4:17%, 5:30%"
4,Female-42kg,RSKC-STOF,Fixed (as published),0.4416,500,397,0.1721,0.2321,0.2695,0.0025,0.0020,
5,Female-42kg,RSKC-STOF,Full pipeline,0.4416,500,500,0.1756,0.2209,0.2659,0.0020,0.0020,"2:24%, 3:27%, 4:27%, 5:22%"
6,Female-42kg,ARSK,Fixed (as published),0.4393,500,234,0.1036,0.2077,0.3357,0.0043,0.0020,
7,Female-42kg,ARSK,Full pipeline,0.4393,500,500,0.1677,0.2118,0.2716,0.0020,0.0020,"2:30%, 3:15%, 4:21%, 5:34%"
8,Female-42kg,DBSCAN,Fixed (as published),0.4768,500,0,NaN,NaN,NaN,NaN,0.0020,
9,Female-42kg,DBSCAN,Full pipeline,0.4768,500,210,0.2837,0.3321,0.1931,0.0047,0.0020,"2:100%, 3:0%"


In [ ]:
# @title Table 8. Permutation Test of Non-random Structure
null_label = {"Full pipeline": "full", "Fixed (as published)": "fixed"}
keys = ["Weight class", "Method"]

d = null_test_df.copy()
d["_null"] = d["Null"].map(null_label)

# --- 1. Descriptive columns from the full-pipeline null ---------------------
full = (d[d["_null"] == "full"]
          .set_index(keys)[["Observed sil.", "Null median", "Null 95th pct.", "Admissible null"]])

# --- 2. p-values side by side (admissible denominator, manuscript convention)
pvals = d.pivot_table(index=keys, columns="_null", values="p (admissible)", aggfunc="first")
pvals = pvals.reindex(columns=["full", "fixed"])
pvals.columns = ["p (full)", "p (fixed)"]

table8 = full.join(pvals).reset_index()

# --- 3. Order: weight class numerically, methods as in METHODS -------------
table8["Method"] = pd.Categorical(table8["Method"], METHODS, ordered=True)
table8["_wc"] = table8["Weight class"].str.extract(r"(\d+)").astype(int)
table8 = (table8.sort_values(["_wc", "Method"])
                .drop(columns="_wc")
                .reset_index(drop=True))
table8["Method"] = table8["Method"].astype(str)

# --- 4. Formatting ----------------------------------------------------------
def fmt3(x):                       # silhouettes: 0.442
    return "–" if pd.isna(x) else f"{x:.3f}"

def fmt_p(x):                      # p-values: 0.002, 0.049, 0.01
    return "–" if pd.isna(x) else f"{round(float(x), 3):g}"

for c in ["Observed sil.", "Null median", "Null 95th pct."]:
    table8[c] = table8[c].map(fmt3)
for c in ["p (full)", "p (fixed)"]:
    table8[c] = table8[c].map(fmt_p)
table8["Admissible null"] = table8["Admissible null"].astype(int)

B = int(null_test_df["B"].iloc[0])
table8["Weight class"] = table8["Weight class"].str.replace(r"(\d+)\s*kg", r"\1 kg", regex=True)
table8 = table8.rename(columns={
    "Weight class": "Class Weight",
    "Observed sil.": "Observed Silhouette",
    "Admissible null": f"Admissible null (of {B})",
})

# --- 5. Display version (merged-cell look) ----------------------------------
table8_display = table8.copy()
table8_display.loc[table8_display["Class Weight"].duplicated(), "Class Weight"] = ""
table8_display

,Class Weight,Method,Observed Silhouette,Null median,Null 95th pct.,Admissible null (of 500),p (full),p (fixed)
0,Female-42 kg,Ward,0.442,0.276,0.321,489,0.002,0.003
1,,k-Means,0.437,0.243,0.321,500,0.002,0.002
2,,RSKC-STOF,0.442,0.176,0.221,500,0.002,0.003
3,,ARSK,0.439,0.168,0.212,500,0.002,0.004
4,,DBSCAN,0.477,0.284,0.332,210,0.005,–
5,Female-47 kg,Ward,0.307,0.218,0.305,450,0.049,0.006
6,,k-Means,0.356,0.213,0.308,500,0.01,0.005
7,,RSKC-STOF,0.319,0.162,0.209,500,0.002,0.005
8,,ARSK,0.319,0.162,0.212,500,0.002,0.006
9,,DBSCAN,0.364,0.286,0.331,36,0.027,0.071


## Re-selection under Resampling

Membership Stability under Bootstrap and Leave-One-Athlete-Out (LOAO) Resampling.

In [ ]:
stability = pd.read_csv(OUTPUT_DIR / 'table5_stability_fixed_vs_pipeline.csv')
stability.head()

,Weight class,Method,Model,Bootstrap admiss. (%),Bootstrap ARI median,ARI 5th pct.,LOAO admiss. (%),LOAO ARI median,Minimum ARI
0,Female-42kg,Ward,Fixed (as published),62.0,0.910,0.598,85.0,1.0,0.898
1,Female-42kg,Ward,Full pipeline,100.0,0.853,0.534,100.0,1.0,0.638
2,Female-42kg,k-Means,Fixed (as published),64.2,0.941,0.603,85.0,1.0,0.938
3,Female-42kg,k-Means,Full pipeline,100.0,0.869,0.572,100.0,1.0,0.695
4,Female-42kg,RSKC-STOF,Fixed (as published),58.0,0.758,0.430,85.0,1.0,0.357


In [ ]:
# @title Table 9. Membership Stability under Bootstrap and Leave-One-Athlete-Out (LOAO) Resampling.

# metric column in CSV  ->  (top header, sub header)
metric_map = {
    'Bootstrap admiss. (%)': ('Bootstrap', 'Admissible (%)'),
    'Bootstrap ARI median':  ('Bootstrap', 'ARI median'),
    'ARI 5th pct.':          ('Bootstrap', 'ARI 5th pct.'),
    'LOAO admiss. (%)':      ('LOAO', 'Admissible (%)'),
    'LOAO ARI median':       ('LOAO', 'ARI median'),
    'Minimum ARI':           ('LOAO', 'Minimum ARI'),
}
method_order = ['Ward', 'k-Means', 'RSKC-STOF', 'ARSK', 'DBSCAN']

s = stability.copy()
s['Model'] = s['Model'].replace({'Fixed (as published)': 'Fixed',
                                 'Full pipeline': 'Full'})
s['Method'] = pd.Categorical(s['Method'], categories=method_order, ordered=True)

# long -> wide: rows = (Weight class, Method), cols = (metric, Fixed/Full)
long = s.melt(id_vars=['Weight class', 'Method', 'Model'],
              value_vars=list(metric_map), var_name='metric')
wide = long.pivot_table(index=['Weight class', 'Method'],
                        columns=['metric', 'Model'],
                        values='value', observed=True)

# Δ = Fixed − Full (same sign convention as the manuscript: 62 − 100 = −38)
for m in metric_map:
  wide[(m, 'Δ')] = wide[(m, 'Full')] - wide[(m, 'Fixed')]

# order columns and build the 3-level header
cols = [(m, k) for m in metric_map for k in ['Fixed', 'Full', 'Δ']]
wide = wide[cols]
wide.columns = pd.MultiIndex.from_tuples(
    [(*metric_map[m], k) for m, k in cols],
    names=['Scheme', 'Metric', 'Model'])
wide = wide.sort_index()

# display formatting: % with 1 decimal, ARI with 3, Δ signed
fmt = {}
for c in wide.columns:
    dec = 1 if c[1] == 'Admissible (%)' else 3
    fmt[c] = f'{{:+.{dec}f}}' if c[2] == 'Δ' else f'{{:.{dec}f}}'

stability_df = wide.style.format(fmt, na_rep='–')
stability_df

In [ ]:
consensus = pd.read_csv(OUTPUT_DIR / 'table8_consensus_fixed_vs_pipeline.csv')
consensus.head()

,Weight class,Method,Model,Admiss. (%),Within,Between,Gap,PAC
0,Female-42kg,Ward,Fixed (as published),32.4,0.953,0.013,0.940,0.047
1,Female-42kg,Ward,Full pipeline,100.0,0.962,0.154,0.808,0.273
2,Female-42kg,k-Means,Fixed (as published),42.0,0.971,0.008,0.963,0.033
3,Female-42kg,k-Means,Full pipeline,100.0,0.995,0.079,0.917,0.172
4,Female-42kg,RSKC-STOF,Fixed (as published),37.0,0.827,0.039,0.787,0.287


In [ ]:
# @title Table 10. Consensus Stability under 80% Subsampling.

FIXED_LABEL, FULL_LABEL = "Fixed (as published)", "Full pipeline"
METHOD_ORDER = ["Ward", "k-Means", "RSKC-STOF", "ARSK", "DBSCAN"]
ADM_COL = next(c for c in consensus.columns if c.startswith("Admiss"))   # "Admissible (%)"
METRICS = [ADM_COL, "Within", "Between", "Gap", "PAC"]               # reorder here if needed

def delta(fixed, full):
    """Δ = Full − Fixed (absolute difference)."""
    return np.nan if pd.isna(fixed) or pd.isna(full) else full - fixed

def consensus_block(wc, metrics):
    """Wide table: rows = methods, columns = (metric, Fixed | Full | Δ)."""
    sub = consensus[consensus["Weight class"] == wc]
    fx = sub[sub["Model"] == FIXED_LABEL].set_index("Method")
    fu = sub[sub["Model"] == FULL_LABEL].set_index("Method")
    methods = [m for m in METHOD_ORDER if m in fx.index or m in fu.index]
    cols = {}
    for metric in metrics:
        f = fx[metric].reindex(methods)
        u = fu[metric].reindex(methods)
        cols[(metric, "Fixed")] = f
        cols[(metric, "Full")] = u
        cols[(metric, "Δ")] = pd.Series([delta(a, b) for a, b in zip(f, u)], index=methods)
    out = pd.DataFrame(cols)
    out.columns = pd.MultiIndex.from_tuples(out.columns, names=[None, "Method"])
    out.index.name = None      # "Method" sits in the header corner -> no empty header row
    return out

def show_block(df, title):
    fmt = {}
    for metric, sub in df.columns:
        dec = 1 if metric == ADM_COL else 3
        fmt[(metric, sub)] = ("{:+.%df}" if sub == "Δ" else "{:.%df}") % dec
    sty = (df.style.format(fmt, na_rep="—")
             .set_caption(title)
             .set_table_styles([
                 {"selector": "caption", "props": "font-weight:bold; font-size:13px; text-align:center;"},
                 {"selector": "th", "props": "text-align:center; border:1px solid #888;"},
                 {"selector": "td", "props": "text-align:center; border:1px solid #888; padding:2px 10px;"}]))
    display(sty)

table10_by_class = {}
for wc in raw_data:
    wc_label = wc.replace("-", "")                     # "Female42kg"
    table10_by_class[wc] = consensus_block(wc, METRICS)
    show_block(table10_by_class[wc], f"Table 10. Consensus Stability under 80% Subsampling of P₀ Partitions – {wc_label}")
    print()

In [ ]:
METHODS = ["Ward", "k-Means", "RSKC-STOF", "ARSK", "DBSCAN"]

def _k_of(sel):
    return int(sel["k"] if "k" in sel else sel["n_clusters"])

def p0_settings_for(wc):
    sel, tp = all_selected[wc], tuned_parameters[wc]
    out = {}
    for m in METHODS:
        s = sel[m]
        if s is None:
            out[m] = None
            continue
        k = _k_of(s)
        st = {"k": k}
        if m in ("k-Means", "RSKC-STOF", "ARSK"):
            st["seed"] = int(s["seed"])
        if m == "RSKC-STOF":
            st["s"] = float(tp["RSKC-STOF s"][k])
        if m == "ARSK":
            st["lambda1"], st["lambda2"] = map(float, tp["ARSK (lambda1, lambda2)"][k])
        if m == "DBSCAN":
            st = {"eps": float(s["eps"]), "minpts": int(s["minpts"]), "k": k}
        out[m] = st
    return out

def _kdist(ks):
    ks = pd.Series(ks).dropna().astype(int)
    if not len(ks):
        return "—"
    vc = ks.value_counts(normalize=True).sort_index() * 100
    return ", ".join(f"{k}:{v:.0f}%" for k, v in vc.items())

In [ ]:
settings_by_class = {wc: p0_settings_for(wc) for wc in raw_data}
print(settings_by_class.keys())
print(settings_by_class['Female-42kg'].keys())
print(settings_by_class['Female-42kg']['ARSK'].keys())

dict_keys(['Female-42kg', 'Female-47kg', 'Female-54kg'])
dict_keys(['Ward', 'k-Means', 'RSKC-STOF', 'ARSK', 'DBSCAN'])
dict_keys(['k', 'seed', 'lambda1', 'lambda2'])


In [ ]:
stab_long = pd.read_csv(OUTPUT_DIR / 'resampling_long.csv')
stab_long.head()

,analysis,weight_class,method,mode,rep,left_out,admissible,k,ARI_cond,ARI_cov,ARI_ret,s,lambda1,lambda2,eps,minpts
0,bootstrap,Female-42kg,Ward,fixed,0,NaN,False,4.0,1.000000,1.000000,1.000000,NaN,NaN,NaN,NaN,NaN
1,bootstrap,Female-42kg,Ward,reselected,0,NaN,True,2.0,0.536026,0.536026,0.536026,NaN,NaN,NaN,NaN,NaN
2,bootstrap,Female-42kg,k-Means,fixed,0,NaN,False,4.0,0.939384,0.939384,0.939384,NaN,NaN,NaN,NaN,NaN
3,bootstrap,Female-42kg,k-Means,reselected,0,NaN,True,2.0,0.598632,0.598632,0.598632,NaN,NaN,NaN,NaN,NaN
4,bootstrap,Female-42kg,RSKC-STOF,fixed,0,NaN,False,4.0,1.000000,1.000000,1.000000,NaN,NaN,NaN,NaN,NaN


In [ ]:
# @title Table 11. Re-selection of the Number of Clusters under Resampling.
def _p0k(wc, m):
    s = all_selected[wc][m]
    return None if s is None else _k_of(s)

def _param_line(g, m, wc):
    st = settings_by_class[wc][m]
    g = g[g["k"] == _p0k(wc, m)]
    if not len(g) or st is None:
        return ""
    if m == "RSKC-STOF":
        return f"s = P0 in {100 * np.mean(np.isclose(g['s'], st['s'])):.0f}%"
    if m == "DBSCAN":
        hit = np.isclose(g["eps"], st["eps"]) & (g["minpts"] == st["minpts"])
        return f"(ε, MinPts) = P0 in {100 * hit.mean():.0f}%"
    if m == "ARSK":
        return (f"λ1 median {g['lambda1'].median():.2f} (P0 {st['lambda1']:.2f}); "
                f"λ2 median {g['lambda2'].median():.1f} (P0 {st['lambda2']:.1f})")
    return ""

cls_resample = []
for analysis in ["bootstrap", "subsample", "loao"]:
    for wc in raw_data:
        for m in METHODS:
            if all_selected[wc][m] is None:
                continue
            g = stab_long.query("analysis == @analysis and weight_class == @wc "
                                "and method == @m and mode == 'reselected'")
            a = g[g["admissible"]]
            cls_resample.append({"Analysis": analysis, "Weight class": wc, "Method": m,
                       "P0 k": _p0k(wc, m), "Replicates": len(g),
                       "No admissible solution (%)": 100 - 100 * g["admissible"].mean(),
                       "P0 k reselected (%)": 100 * (g["k"] == _p0k(wc, m)).mean(),
                       "k distribution": _kdist(a["k"]),
                       "Tuning parameters at P0 k": _param_line(a, m, wc),
                       "Median ARI vs P0": a["ARI_ret"].median(),
                       "ARI 5th pct.": a["ARI_ret"].quantile(0.05)})
clusters_resample = pd.DataFrame(cls_resample).round(3)
display(clusters_resample)
print("ARI: full sample, DBSCAN noise as singletons.")

,Analysis,Weight class,Method,P0 k,Replicates,No admissible solution (%),P0 k reselected (%),k distribution,Tuning parameters at P0 k,Median ARI vs P0,ARI 5th pct.
0,bootstrap,Female-42kg,Ward,4,500,0.0,28.8,"2:18%, 3:37%, 4:29%, 5:16%",,0.853,0.534
1,bootstrap,Female-42kg,k-Means,4,500,0.0,28.6,"2:12%, 3:39%, 4:29%, 5:21%",,0.869,0.572
2,bootstrap,Female-42kg,RSKC-STOF,4,500,0.0,34.0,"2:3%, 3:37%, 4:34%, 5:26%",s = P0 in 32%,0.857,0.509
3,bootstrap,Female-42kg,ARSK,4,500,0.0,38.4,"2:7%, 3:36%, 4:38%, 5:18%",λ1 median 1.46 (P0 0.93); λ2 median 41.5 (P0 4...,0.810,0.483
4,bootstrap,Female-42kg,DBSCAN,3,500,3.8,50.0,"2:8%, 3:52%, 4:30%, 5:7%, 6:3%, 7:0%","(ε, MinPts) = P0 in 22%",0.881,0.457
5,bootstrap,Female-47kg,Ward,3,500,0.0,25.2,"2:22%, 3:25%, 4:20%, 5:33%",,0.567,0.323
6,bootstrap,Female-47kg,k-Means,5,500,0.0,49.6,"2:10%, 3:17%, 4:23%, 5:50%",,0.692,0.266
7,bootstrap,Female-47kg,RSKC-STOF,5,500,0.0,70.0,"2:4%, 3:3%, 4:23%, 5:70%",s = P0 in 26%,0.552,0.241
8,bootstrap,Female-47kg,ARSK,5,500,0.0,61.8,"2:1%, 3:8%, 4:29%, 5:62%",λ1 median 1.02 (P0 1.32); λ2 median 41.0 (P0 4...,0.543,0.249
9,bootstrap,Female-47kg,DBSCAN,2,500,16.2,53.0,"2:63%, 3:24%, 4:8%, 5:4%, 6:1%","(ε, MinPts) = P0 in 17%",0.638,0.195


ARI: full sample, DBSCAN noise as singletons.


In [ ]:
import re
import numpy as np
import pandas as pd

df = clusters_resample.copy()

# --- 1. Standardise analysis labels -----------------------------------------
analysis_map = {
    "bootstrap": "Bootstrap",
    "subsample": "Subsample",
    "loao": "LOAO",
}
df["Analysis_std"] = (
    df["Analysis"].astype(str).str.strip().str.lower()
      .map(analysis_map)
      .fillna(df["Analysis"])
)

keys = ["Weight class", "Method"]

# --- 2. P0 k re-selected (%) per analysis, side by side ---------------------
reselect = (
    df.pivot_table(index=keys, columns="Analysis_std",
                   values="P0 k reselected (%)", aggfunc="first")
      .reindex(columns=["Bootstrap", "Subsample", "LOAO"])
)
reselect.columns = [f"P0 k re-selected (%): {c}" for c in reselect.columns]

# --- 3. P0 k (should be identical across analyses) --------------------------
p0k = df.groupby(keys)["P0 k"].first()
assert (df.groupby(keys)["P0 k"].nunique() == 1).all(), "P0 k differs across analyses"

# --- 4. Most selected k from the bootstrap k distribution -------------------
def most_selected_k(dist):
    """'2:18%, 3:37%, 4:29%, 5:16%' -> '3 (37)'. Ties go to the smaller k."""
    if pd.isna(dist):
        return np.nan
    pairs = [(int(k), float(p))
             for k, p in re.findall(r"(\d+)\s*:\s*([\d.]+)\s*%", str(dist))]
    if not pairs:
        return np.nan
    k, pct = max(pairs, key=lambda t: (t[1], -t[0]))
    return f"{k} ({pct:g})"

most_k = (
    df[df["Analysis_std"] == "Bootstrap"]
      .set_index(keys)["k distribution"]
      .map(most_selected_k)
      .rename("Most selected k, bootstrap (%)")
)

# --- 5. Assemble ------------------------------------------------------------
table11 = pd.concat([p0k.rename("P0 k"), reselect, most_k], axis=1).reset_index()

# Order: weight class numerically, methods as in the manuscript
method_order = ["Ward", "k-Means", "RSKC-STOF", "ARSK", "DBSCAN"]
table11["Method"] = pd.Categorical(table11["Method"], method_order, ordered=True)
table11["_wc"] = table11["Weight class"].str.extract(r"(\d+)").astype(int)
table11 = (table11.sort_values(["_wc", "Method"])
                  .drop(columns="_wc")
                  .reset_index(drop=True))

# "Female-42kg" -> "Female 42 kg"
table11["Weight class"] = table11["Weight class"].str.replace(
    r"^(\w+)-(\d+)\s*kg$", r"\1 \2 kg", regex=True)
table11 = table11.rename(columns={"Weight class": "Class Weight"})

# Number formatting: 34.0 -> 34, 28.8 -> 28.8, 100.0 -> 100
pct_cols = [c for c in table11.columns if c.startswith("P0 k re-selected")]
table11[pct_cols] = table11[pct_cols].map(lambda x: "" if pd.isna(x) else f"{x:g}")
table11["P0 k"] = table11["P0 k"].astype(int)

# --- 6. Display version with the repeated weight class blanked (merged-cell look)
table11_display = table11.copy()
table11_display.loc[table11_display["Class Weight"].duplicated(), "Class Weight"] = ""

table11_display

,Class Weight,Method,P0 k,P0 k re-selected (%): Bootstrap,P0 k re-selected (%): Subsample,P0 k re-selected (%): LOAO,"Most selected k, bootstrap (%)"
0,Female 42 kg,Ward,4,28.8,30.8,85,3 (37)
1,,k-Means,4,28.6,41.6,85,3 (39)
2,,RSKC-STOF,4,34,30,80,3 (37)
3,,ARSK,4,38.4,27.2,80,4 (38)
4,,DBSCAN,3,50,69.6,95,3 (52)
5,Female 47 kg,Ward,3,25.2,47.6,65,5 (33)
6,,k-Means,5,49.6,32.6,75,5 (50)
7,,RSKC-STOF,5,70,55.2,55,5 (70)
8,,ARSK,5,61.8,48.2,40,5 (62)
9,,DBSCAN,2,53,75.8,95,2 (63)


## Cluster Profiles Beyond Scoring

In [ ]:
# @title Cluster profiles: raw P0 features with 1,000 match-block bootstrap 95% CIs
N_PROFILE_BOOT = 1000
PROFILE_SEED = 20260514

profile_rows = []
for class_index, (wc, methods) in enumerate(all_selected.items()):
    df = raw_data[wc]
    matches = np.array(sorted(df["match"].unique()))
    match_rows = {mid: np.where(df["match"] == mid)[0] for mid in matches}
    rng = np.random.default_rng(PROFILE_SEED + class_index)
    boots = [np.concatenate([match_rows[mid] for mid in rng.choice(matches, len(matches), replace=True)])
             for _ in range(N_PROFILE_BOOT)]           # same resamples for every method
    for name, sel in methods.items():
        if sel is None:
            continue
        lab = np.asarray(sel["labels"])
        for c in sorted(set(lab.tolist()) - {-1}):
            members = lab == c
            for f in P0_FEATURES:
                x = df[f].to_numpy(float)
                bm = [x[idx][members[idx]].mean() for idx in boots if members[idx].any()]
                profile_rows.append({
                    "Weight class": wc, "Method": name, "Cluster": c,
                    "n": int(members.sum()), "Feature": f,
                    "Mean": x[members].mean(), "SD": x[members].std(ddof=1),
                    "CI low": np.percentile(bm, 2.5), "CI high": np.percentile(bm, 97.5),
                    "Zero (%)": 100.0 * np.mean(x[members] == 0)})

profiles = pd.DataFrame(profile_rows).round(3)

# compact view: mean (95% CI) per cluster and feature
profiles["Mean (95% CI)"] = profiles.apply(
    lambda r: f"{r['Mean']:.2f} ({r['CI low']:.2f}–{r['CI high']:.2f})", axis=1)

display(profiles.pivot_table(index=["Weight class", "Method", "Cluster", "n"],
                             columns="Feature", values="Mean (95% CI)", aggfunc="first")
        [P0_FEATURES])

Feature                                        yuko          waza_ari  \
Weight class Method    Cluster n                                        
Female-42kg  ARSK      0       5   1.40 (0.00–2.50)  0.00 (0.00–0.00)   
                       1       13  0.15 (0.00–0.50)  0.00 (0.00–0.00)   
                       2       16  4.69 (3.56–6.06)  0.00 (0.00–0.00)   
                       3       4   4.50 (3.00–6.00)  1.00 (1.00–1.00)   
             DBSCAN    0       13  0.15 (0.00–0.50)  0.00 (0.00–0.00)   
                       1       5   1.40 (0.00–2.50)  0.00 (0.00–0.00)   
                       2       15  4.67 (3.45–6.12)  0.00 (0.00–0.00)   
             RSKC-STOF 0       15  4.67 (3.45–6.12)  0.00 (0.00–0.00)   
                       1       4   4.50 (3.00–6.00)  1.00 (1.00–1.00)   
                       2       5   1.40 (0.00–2.50)  0.00 (0.00–0.00)   
                       3       14  0.50 (0.00–1.40)  0.00 (0.00–0.00)   
             Ward      0       14  0.50 (0.00–1.40)  0.00 (0.00–0.00)   
                       1       5   1.40 (0.00–2.50)  0.00 (0.00–0.00)   
                       2       4   4.50 (3.00–6.00)  1.00 (1.00–1.00)   
                       3       15  4.67 (3.45–6.12)  0.00 (0.00–0.00)   
             k-Means   0       6   2.00 (0.67–3.60)  0.00 (0.00–0.00)   
                       1       15  4.67 (3.45–6.12)  0.00 (0.00–0.00)   
                       2       4   4.50 (3.00–6.00)  1.00 (1.00–1.00)   
                       3       13  0.15 (0.00–0.50)  0.00 (0.00–0.00)   
Female-47kg  ARSK      0       8   2.25 (1.14–3.00)  0.00 (0.00–0.00)   
                       1       8   1.12 (0.70–1.56)  0.00 (0.00–0.00)   
                       2       9   2.44 (1.29–3.70)  0.00 (0.00–0.00)   
                       3       4   2.25 (0.50–6.00)  1.25 (1.00–1.50)   
                       4       9   0.00 (0.00–0.00)  0.00 (0.00–0.00)   
             DBSCAN    0       9   0.00 (0.00–0.00)  0.00 (0.00–0.00)   
                       1       22  2.04 (1.70–2.43)  0.00 (0.00–0.00)   
             RSKC-STOF 0       9   0.00 (0.00–0.00)  0.00 (0.00–0.00)   
                       1       8   2.25 (1.14–3.00)  0.00 (0.00–0.00)   
                       2       9   2.44 (1.29–3.70)  0.00 (0.00–0.00)   
                       3       8   1.12 (0.70–1.56)  0.00 (0.00–0.00)   
                       4       4   2.25 (0.50–6.00)  1.25 (1.00–1.50)   
             Ward      0       9   0.00 (0.00–0.00)  0.00 (0.00–0.00)   
                       1       7   1.57 (0.43–2.86)  0.00 (0.00–0.00)   
                       2       22  2.14 (1.60–2.79)  0.23 (0.00–0.52)   
             k-Means   0       10  1.60 (1.20–2.00)  0.00 (0.00–0.00)   
                       1       8   2.75 (1.60–4.00)  0.00 (0.00–0.00)   
                       2       7   1.57 (0.43–2.86)  0.00 (0.00–0.00)   
                       3       9   0.00 (0.00–0.00)  0.00 (0.00–0.00)   
                       4       4   2.25 (0.50–6.00)  1.25 (1.00–1.50)   
Female-54kg  ARSK      0       25  2.96 (2.14–4.05)  0.00 (0.00–0.00)   
                       1       13  0.00 (0.00–0.00)  0.00 (0.00–0.00)   
             DBSCAN    0       10  0.00 (0.00–0.00)  0.00 (0.00–0.00)   
                       1       21  2.29 (1.71–2.88)  0.00 (0.00–0.00)   
             RSKC-STOF 0       25  2.96 (2.14–4.05)  0.00 (0.00–0.00)   
                       1       13  0.00 (0.00–0.00)  0.00 (0.00–0.00)   
             Ward      0       13  0.00 (0.00–0.00)  0.00 (0.00–0.00)   
                       1       25  2.96 (2.14–4.05)  0.00 (0.00–0.00)   
             k-Means   0       25  2.96 (2.14–4.05)  0.00 (0.00–0.00)   
                       1       13  0.00 (0.00–0.00)  0.00 (0.00–0.00)   

Feature                                       ippon               attack  \
Weight class Method    Cluster n                                           
Female-42kg  ARSK      0       5   2.00 (2.00–2.00)     3.80 (3.00–4.50)   
                       1       13  0.00 (0.00–0.

In [ ]:
# @title Table 12. Cluster Profiles with Bootstrap 95% Confidence Intervals - Ward Algorithm.
# --- Settings ---------------------------------------------------------------
# Which method's clusters to report for each weight class
TABLE12_METHOD = {
    "Female-42kg": "Ward",   # <- change to the method you report
    "Female-47kg": "Ward",
    "Female-54kg": "Ward",
}

# Raw feature name -> manuscript column (order = column order)
FEATURE_LABELS = {
    "yuko": "Yuko",
    "waza_ari": "Waza-ari",
    "ippon": "Ippon",
    "attack": "Attack (95% CI)",
    "defend": "Defend (95% CI)",
    "longest_point_in_sec": "Longest (s)",
    "fastest_point_in_sec": "Fastest (s)",
}
CI_FEATURES = {"attack", "defend"}   # features shown as mean (95% CI)

# --- 1. Keep only the reported method per class -----------------------------
p = profiles[profiles["Method"] == profiles["Weight class"].map(TABLE12_METHOD)].copy()

# --- 2. Format each cell ----------------------------------------------------
def fmt_mean(x):
    """0.500 -> 0.5, 2.000 -> 2, 4.667 -> 4.67"""
    return f"{round(float(x), 2):g}"

p["Cell"] = np.where(
    p["Feature"].isin(CI_FEATURES),
    p.apply(lambda r: f"{r['Mean']:.2f} ({r['CI low']:.2f}–{r['CI high']:.2f})", axis=1),
    p["Mean"].map(fmt_mean),
)

# --- 3. Wide layout: one row per cluster -----------------------------------
table12 = (
    p.pivot_table(index=["Weight class", "Cluster", "n"], columns="Feature",
                  values="Cell", aggfunc="first")
     .reindex(columns=list(FEATURE_LABELS))   # features not in P0 for a class -> NaN
     .fillna("–")
     .reset_index()
)

# Clusters numbered 1, 2, 3 ... within each class (handles 0-based labels)
table12["Cls."] = (table12.groupby("Weight class")["Cluster"]
                          .rank(method="dense").astype(int))

# Sort: weight class numerically, then cluster
table12["_wc"] = table12["Weight class"].str.extract(r"(\d+)").astype(int)
table12 = table12.sort_values(["_wc", "Cls."]).reset_index(drop=True)

# "Female-42kg" -> "Female 42 kg", rename columns, final order
table12["Weight class"] = table12["Weight class"].str.replace(
    r"^(\w+)-(\d+)\s*kg$", r"\1 \2 kg", regex=True)
table12 = (table12.rename(columns={"Weight class": "Class Weight", **FEATURE_LABELS})
                  [["Class Weight", "Cls.", "n", *FEATURE_LABELS.values()]])

# --- 4. Display version (merged-cell look) ----------------------------------
table12_display = table12.copy()
table12_display.loc[table12_display["Class Weight"].duplicated(), "Class Weight"] = ""
table12_display

Feature,Class Weight,Cls.,n,Yuko,Waza-ari,Ippon,Attack (95% CI),Defend (95% CI),Longest (s),Fastest (s)
0,Female 42 kg,1,14,0.5,0,0.07,6.29 (5.00–7.77),2.64 (1.83–3.47),0,0
1,,2,5,1.4,0,2,3.80 (3.00–4.50),5.00 (3.33–6.33),6,3
2,,3,4,4.5,1,0.25,6.25 (5.00–9.00),3.50 (1.00–5.00),6.5,3.5
3,,4,15,4.67,0,0,5.13 (4.36–6.00),3.93 (3.21–4.73),5.47,3.33
4,Female 47 kg,1,9,0,0,0,7.22 (5.43–9.00),2.44 (1.64–3.33),0,0
5,,2,7,1.57,0,1.57,8.29 (5.00–12.00),5.14 (2.60–8.00),4,2.86
6,,3,22,2.14,0.23,0.04,9.00 (7.16–10.87),4.32 (3.18–5.58),2.23,2.04
7,Female 54 kg,1,13,0,0,0,5.85 (5.50–6.20),3.08 (2.17–3.92),0,0
8,,2,25,2.96,0,0,6.04 (5.57–6.48),3.64 (3.19–4.11),4.92,2.8
